## LIBRARY SETUP

In [ ]:
!pip install cadquery
!pip install vedo
!pip install tiktoken
!pip install jsonschema
!pip install pymupdf


import tensorflow as tf
import numpy as np
import scipy
import jax
import flax
import jaxlib
import optax
import scipy
import time
import tempfile
import numpy as np
import math
import vedo
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import cadquery as cq
from IPython.display import display, Markdown,Image, clear_output
from google.colab import output, userdata
from vedo import Mesh, show, screenshot
import openai
import os
import base64
from openai import OpenAI
import tiktoken
import json
import re
from jsonschema import validate, ValidationError
from collections import defaultdict
import pandas as pd
from typing import List, Dict, Tuple
import pathlib
import ast


CAD VIEWS

In [ ]:
import os
import fitz  # PyMuPDF
import base64
import openai
from IPython.display import Image, display
from PIL import Image as PILImage

# --- SETUP ---
PDF_FILE = "viste piastra.pdf"
assert os.path.exists(PDF_FILE), "Il file PDF non è stato trovato."

# --- CONVERSIONE PDF IN IMMAGINI ---
pdf_doc = fitz.open(PDF_FILE)
image_paths = []

for i, page in enumerate(pdf_doc):
    pix = page.get_pixmap(dpi=300)  # alta qualità
    output_path = f"page_{i+1}.png"
    pix.save(output_path)
    image_paths.append(output_path)

print("✅ Immagini estratte:", image_paths)

# --- FUNZIONE PER BASE64 ---
def load_image_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# --- CHIAMATA A GPT-4o-mini ---
def describe_geometry_from_pdf_images(image_paths, model="gpt-4o"):

    prompt = """
    You are given one or more orthographic technical views and sectional views of a mechanical part extracted from a SolidWorks PDF drawing. The drawing includes explicit dimensional annotations.

    TASK:
    Provide a concise but comprehensive geometric description of the part in natural language using clear, technical language. Your description should explicitly reference the dimensions provided in the drawing. Include the following details whenever clearly indicated:

    - Overall shape and primary dimensions (length, width, thickness), explicitly citing values from the drawing.
    - Identification, positioning, and dimensions of all visible features (e.g., holes, slots, pockets, chamfers, fillets).
    - Clearly indicate whether features like holes or pockets are through or blind. For blind features, explicitly state their depth as given by sectional views.
    - Specify clearly on which face each feature is located (e.g., "top face", "bottom face", "side face"), and provide their positions explicitly using given dimensions by the. If a feature is centered on a face, clearly state that it is centered (e.g., "the hole is centered on the top face").
    - do not ignore any geometric dimeesions in the pdf

    Do NOT:
    - Invent or guess features not explicitly visible or dimensioned.
    - Provide manufacturing instructions, tooling advice, or markup annotations.

    Your output will be used directly for CNC machining process planning.

    Output ONLY the geometric description text.
    """

    messages = [
        {"role": "system", "content": "You are a mechanical engineer interpreting CAD drawings."},
        {"role": "user", "content": prompt}
    ]

    for img_path in image_paths:
        img_b64 = load_image_base64(img_path)
        messages.append({
            "role": "user",
            "content": [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}]
        })

    openai_api_key = userdata.get('OpenAI_API')
    openai.api_key = openai_api_key

    # Chiamata all'LLM
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.1,
        max_tokens=3000
    )

    return response.choices[0].message.content.strip()

# --- ESECUZIONE ---
desc = describe_geometry_from_pdf_images(image_paths)
print("📐 DESCRIZIONE GEOMETRICA:")
print(desc)


AssertionError: Il file PDF non è stato trovato.

GEOMETRY FROM USER DESCRIPTION


In [ ]:
def transform_local_to_world(point_local, origin, rotation_deg):
    rx, ry, rz = np.radians(rotation_deg)

    # Rotation around X
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)]
    ])

    # Rotation around Y
    Ry = np.array([
        [np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])

    # Rotation around Z
    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0, 0, 1]
    ])

    # Total composition (intrinsic ZYX)
    R = Rz @ Ry @ Rx

    point_world = R @ point_local + origin
    return point_world, R

In [ ]:
def plot_plate_3d(dimensions, feature=None):
    length = dimensions["length_mm"]
    width = dimensions["width_mm"]
    thickness = dimensions["thickness_mm"]
    origin_offset = np.array(dimensions.get("origin_mm", [0, 0, 0]))
    rotation_deg = dimensions.get("rotation_deg", [0, 0, 0])

    # Coordinate bounds centered around origin
    x0, x1 = -length / 2, length / 2
    y0, y1 = -width / 2, width / 2
    z0, z1 = -thickness / 2, thickness / 2

    # Define 8 corners of the plate (centered at origin)
    corners = np.array([
        [x0, y0, z0],
        [x1, y0, z0],
        [x1, y1, z0],
        [x0, y1, z0],
        [x0, y0, z1],
        [x1, y0, z1],
        [x1, y1, z1],
        [x0, y1, z1]
    ])

    # Define faces
    faces = [
        [corners[0], corners[1], corners[2], corners[3]],  # bottom
        [corners[4], corners[5], corners[6], corners[7]],  # top
        [corners[0], corners[1], corners[5], corners[4]],  # front
        [corners[1], corners[2], corners[6], corners[5]],  # right
        [corners[2], corners[3], corners[7], corners[6]],  # back
        [corners[3], corners[0], corners[4], corners[7]]   # left
    ]

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    for face in faces:
        ax.add_collection3d(
            Poly3DCollection([face], facecolors='gray', edgecolors='k', alpha=0.3)
        )

    # Add reference axes centered at origin
    arrow_len = max(length, width, thickness) * 0.6
    ax.quiver(0, 0, 0, arrow_len, 0, 0, color='blue', arrow_length_ratio=0.05)
    ax.text(arrow_len * 1.02, 0, 0, 'X', color='blue')
    ax.quiver(0, 0, 0, 0, arrow_len, 0, color='blue', arrow_length_ratio=0.05)
    ax.text(0, arrow_len * 1.02, 0, 'Y', color='blue')
    ax.quiver(0, 0, 0, 0, 0, arrow_len, color='blue', arrow_length_ratio=0.05)
    ax.text(0, 0, arrow_len, 'Z', color='blue')

    direction_global = [0, 0, 0]
    pos_global = [0, 0, 0]
    if feature is not None:
        # Extend dimensions to pass origin and rotation
        dim_extended = dimensions.copy()
        dim_extended["origin_mm"] = origin_offset
        dim_extended["rotation_deg"] = rotation_deg
        direction_global, pos_global = plot_feature_direction_arrow(ax, dim_extended, feature)

    # Set aspect ratio and view
    ax.set_box_aspect([length, width, thickness])
    ax.view_init(elev=30, azim=45)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    time.sleep(0.3)
    return direction_global, pos_global



def plot_feature_direction_arrow(ax, dimensions, feature, length=30.0, color='red'):
    origin_offset = np.array(dimensions["origin_mm"])
    rotation_deg = dimensions["rotation_deg"]
    pos_local = np.array(feature["position_mm"])
    direction_info = feature["machining_direction"]

    # Base direction in user system
    if direction_info["type"] == "main_axis":
        dir_dict = {
            'X': np.array([1.0, 0.0, 0.0]),
            'Y': np.array([0.0, 1.0, 0.0]),
            'Z': np.array([0.0, 0.0, 1.0]),
        }
        direction_local = dir_dict.get(direction_info["reference_axis"].upper(), np.array([0.0, 0.0, 1.0]))
        direction_local *= direction_info.get("direction", 1)

    elif direction_info["type"] == "inclined_direction":
        angle_xy = np.radians(direction_info["angle_xy_deg"])
        angle_z = np.radians(direction_info["angle_z_deg"])
        sign = direction_info.get("direction", 1)

        r_xy = np.cos(angle_z)
        dx = r_xy * np.cos(angle_xy)
        dy = r_xy * np.sin(angle_xy)
        dz = np.sin(angle_z)
        direction_local = np.array([dx, dy, dz])
        direction_local *= sign
        direction_local /= np.linalg.norm(direction_local)

    else:
        raise ValueError("Invalid machining direction format.")

    # Transfer position and direction in global system
    pos_global, R = transform_local_to_world(pos_local, origin_offset, rotation_deg)
    direction_global = R @ direction_local

    # Draw arrow
    ax.quiver(*pos_global, *(direction_global * length), color=color, arrow_length_ratio=0.1)
    ax.text(*(pos_global + direction_global * length * 1.05), 'Hole', color=color)
    return direction_global, pos_global


dimensions = {
    "length_mm": 120.0,
    "width_mm": 80.0,
    "thickness_mm": 10.0,
    "origin_mm": [0, 0, 0],
    "rotation_deg": [0, 0, 0],
}

feature = {
    "diameter_mm": 10,
    "position_mm": [30, 20, 5],
    "machining_direction": {
        "type": "main_axis",
        "reference_axis": "Z",
        "direction": -1
    }
}

plot_plate_3d(dimensions, feature)

In [ ]:
def plot_cylinder_3d(dimensions,feature=None):
    radius = dimensions["diameter_mm"] / 2
    height = dimensions["height_mm"]
    origin = dimensions["origin_mm"]
    n_points = 100
    theta = np.linspace(0, 2 * np.pi, n_points)

    # Coordinates for top and bottom circles
    x = radius * np.cos(theta)
    y = radius * np.sin(theta)
    z_bottom = np.zeros_like(x)
    z_top = np.ones_like(x) * height

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    # Side faces
    for i in range(n_points - 1):
        verts = [[
            [x[i], y[i], 0],
            [x[i + 1], y[i + 1], 0],
            [x[i + 1], y[i + 1], height],
            [x[i], y[i], height]
        ]]
        ax.add_collection3d(Poly3DCollection(verts, facecolors='gray', edgecolors='none', alpha=0.3))

    # Closing the loop (last segment)
    verts = [[
        [x[-1], y[-1], 0],
        [x[0], y[0], 0],
        [x[0], y[0], height],
        [x[-1], y[-1], height]
    ]]
    ax.add_collection3d(Poly3DCollection(verts, facecolors='gray', edgecolors='none', alpha=0.3))

    # Top and bottom surfaces
    # Bottom face — clean ring (no radial lines)
    bottom_ring = [[x[i], y[i], 0] for i in range(n_points)]
    ax.add_collection3d(Poly3DCollection([bottom_ring], facecolors='gray', edgecolors='k', alpha=0.3, closed=True))

    # Top face — clean ring (no radial lines)
    top_ring = [[x[i], y[i], height] for i in range(n_points)]
    ax.add_collection3d(Poly3DCollection([top_ring], facecolors='gray', edgecolors='k', alpha=0.3, closed=True))

    # Arrow origin and length
    arrow_color = 'blue'
    arrow_length = dimensions["diameter_mm"] * 1.2
    origin = np.array(dimensions["origin_mm"])  # [x0, y0, z0]
    rotation_deg = dimensions["rotation_deg"]   # [rx, ry, rz]
    rx, ry, rz = np.radians(rotation_deg)        # convert to radians

    # Define rotation matrices (right-hand rule, intrinsic rotations)
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)]
    ])

    Ry = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [0,           1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])

    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0,           0,          1]
    ])

    # Combined rotation matrix: R = Rz * Ry * Rx
    R = Rz @ Ry @ Rx

    # Local axes (unit vectors)
    X_local = R @ np.array([1, 0, 0]) * arrow_length
    Y_local = R @ np.array([0, 1, 0]) * arrow_length
    Z_local = R @ np.array([0, 0, 1]) * arrow_length

    # Draw arrows from origin
    ax.quiver(*origin, *X_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + X_local * 1.02), 'X', color=arrow_color)

    ax.quiver(*origin, *Y_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + Y_local * 1.02), 'Y', color=arrow_color)

    ax.quiver(*origin, *Z_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + Z_local * 1.02), 'Z', color=arrow_color)

    direction_global=[0,0,0]
    pos_global=[0,0,0]
    if feature is not None:
       direction_global, pos_global = plot_feature_direction_arrow(ax, dimensions, feature)

    # Set aspect ratio and view
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=30, azim=45)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    time.sleep(0.3)
    return direction_global, pos_global

def plot_feature_direction_arrow(ax, dimensions, feature, length=30.0, color='red'):
    origin_offset = np.array(dimensions["origin_mm"])
    rotation_deg = dimensions["rotation_deg"]
    pos_local = np.array(feature["position_mm"])
    direction_info = feature["machining_direction"]

    # Base direction in user system
    if direction_info["type"] == "main_axis":
        dir_dict = {
            'X': np.array([1.0, 0.0, 0.0]),
            'Y': np.array([0.0, 1.0, 0.0]),
            'Z': np.array([0.0, 0.0, 1.0]),
        }
        direction_local = dir_dict.get(direction_info["reference_axis"].upper(), np.array([0.0, 0.0, 1.0]))
        direction_local *= direction_info.get("direction", 1)

    elif direction_info["type"] == "inclined_direction":
        angle_xy = np.radians(direction_info["angle_xy_deg"])
        angle_z = np.radians(direction_info["angle_z_deg"])
        sign = direction_info.get("direction", 1)

        r_xy = np.cos(angle_z)
        dx = r_xy * np.cos(angle_xy)
        dy = r_xy * np.sin(angle_xy)
        dz = np.sin(angle_z)
        direction_local = np.array([dx, dy, dz])
        direction_local *= sign
        direction_local /= np.linalg.norm(direction_local)

    else:
        raise ValueError("Invalid machining direction format.")

    # Transfer position and direction in global system
    pos_global, R = transform_local_to_world(pos_local, origin_offset, rotation_deg)
    direction_global = R @ direction_local

    # Draw arrow
    ax.quiver(*pos_global, *(direction_global * length), color=color, arrow_length_ratio=0.1)
    ax.text(*(pos_global + direction_global * length * 1.05), 'Hole', color=color)
    return direction_global, pos_global

dimensions = {
    "diameter_mm": 80,
    "height_mm": 100,
    "origin_mm": [0, 0, 0],
    "rotation_deg": [0, 0, 0],}
     # ruota attorno
feature = {
        "diameter_mm": 20,
       "position_mm": [0, 0, 100],
        "machining_direction":{"type": "inclined_direction", "angle_xy_deg": 70, "angle_z_deg": 0, "sign": 1}
       #"machining_direction": {"type": "main_axis", "direction": "Z", "sign": -1}
    }



plot_cylinder_3d(dimensions,feature)

In [ ]:
def plot_gear_3d(dimensions,feature=None):
    num_teeth = dimensions["number_of_teeth"]
    outer_radius = dimensions["outer_diameter"] / 2
    root_radius = dimensions["root_diameter"] / 2
    top_width = dimensions["tooth_top_width"]
    base_width = dimensions["tooth_base_width"]
    tooth_height = dimensions["tooth_height"]
    thickness = dimensions["gear_thickness"]
    clearance_angle_deg = dimensions["tooth_clearance_angle"]

    angle_step = 2 * np.pi / num_teeth

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    for i in range(num_teeth):
        angle = i * angle_step

        r_base = root_radius
        r_tip = outer_radius
        center_base = np.array([r_base * np.cos(angle), r_base * np.sin(angle)])
        center_tip = np.array([r_tip * np.cos(angle), r_tip * np.sin(angle)])

        tangent = np.array([-np.sin(angle), np.cos(angle)])
        tangent /= np.linalg.norm(tangent)

        top_half = top_width / 2
        base_half = base_width / 2

        v0 = center_base - tangent * base_half
        v1 = center_base + tangent * base_half
        v2 = center_tip + tangent * top_half
        v3 = center_tip - tangent * top_half

        v0_up = np.append(v0, thickness)
        v1_up = np.append(v1, thickness)
        v2_up = np.append(v2, thickness)
        v3_up = np.append(v3, thickness)

        v0 = np.append(v0, 0)
        v1 = np.append(v1, 0)
        v2 = np.append(v2, 0)
        v3 = np.append(v3, 0)

        top_face = [v0, v1, v2, v3]
        bottom_face = [v0_up, v1_up, v2_up, v3_up]
        side1 = [v0, v0_up, v3_up, v3]
        side2 = [v1, v1_up, v2_up, v2]
        front = [v3, v2, v2_up, v3_up]

        ax.add_collection3d(Poly3DCollection([top_face], facecolors='lightgray', edgecolors='none',alpha=0.3))
        ax.add_collection3d(Poly3DCollection([bottom_face], facecolors='lightgray', edgecolors='none',alpha=0.3))
        ax.add_collection3d(Poly3DCollection([side1], facecolors='gray', edgecolors='k',alpha=0.3))
        ax.add_collection3d(Poly3DCollection([side2], facecolors='gray', edgecolors='k',alpha=0.3))
        ax.add_collection3d(Poly3DCollection([front], facecolors='dimgray', edgecolors='k',alpha=0.3))

    # Add central cylinder (solid disc)
    n_circle = 100
    theta = np.linspace(0, 2 * np.pi, n_circle)
    r_inner = 0.0
    r_outer = root_radius

    # Lower circle (z = 0)
    x_lower = r_outer * np.cos(theta)
    y_lower = r_outer * np.sin(theta)
    verts_lower = [[0, 0, 0]] + [[x, y, 0] for x, y in zip(x_lower, y_lower)]

    # Upper circle (z = thickness)
    x_upper = r_outer * np.cos(theta)
    y_upper = r_outer * np.sin(theta)
    verts_upper = [[0, 0, thickness]] + [[x, y, thickness] for x, y in zip(x_upper, y_upper)]

    # Side wall of the cylinder
    for i in range(n_circle - 1):
        x0, y0 = x_lower[i], y_lower[i]
        x1, y1 = x_lower[i + 1], y_lower[i + 1]
        ax.add_collection3d(Poly3DCollection(
            [[
                [x0, y0, 0],
                [x1, y1, 0],
                [x1, y1, thickness],
                [x0, y0, thickness]
            ]],
            facecolors='gray', edgecolors='none',alpha=0.3
        ))

    # Close the loop
    x0, y0 = x_lower[-1], y_lower[-1]
    x1, y1 = x_lower[0], y_lower[0]
    ax.add_collection3d(Poly3DCollection(
        [[
            [x0, y0, 0],
            [x1, y1, 0],
            [x1, y1, thickness],
            [x0, y0, thickness]
        ]],
        facecolors='gray', edgecolors='none',alpha=0.3
    ))

    # Create only the outer ring to avoid radial lines
    ring_lower = [v[:3] for v in verts_lower[1:]]  # exclude center point [0, 0, 0]
    ring_upper = [v[:3] for v in verts_upper[1:]]  # exclude center point [0, 0, height]

    ax.add_collection3d(Poly3DCollection([ring_lower], facecolors='gray', edgecolors='k', alpha=0.3, closed=True))
    ax.add_collection3d(Poly3DCollection([ring_upper], facecolors='gray', edgecolors='k', alpha=0.3, closed=True))

    arrow_color = 'blue'
    arrow_length = dimensions["root_diameter"] * 0.75
    origin = np.array(dimensions["origin_mm"])
    rx, ry, rz = np.radians(dimensions["rotation_deg"])

    # Rotation matrices
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)]
    ])

    Ry = np.array([
        [np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])

    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0, 0, 1]
    ])

    # Combined rotation
    R = Rz @ Ry @ Rx

    # Transform local axes
    X_local = R @ np.array([1, 0, 0]) * arrow_length
    Y_local = R @ np.array([0, 1, 0]) * arrow_length
    Z_local = R @ np.array([0, 0, 1]) * arrow_length

    # Plot
    ax.quiver(*origin, *X_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + X_local * 1.02), 'X', color=arrow_color)

    ax.quiver(*origin, *Y_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + Y_local * 1.02), 'Y', color=arrow_color)

    ax.quiver(*origin, *Z_local, color=arrow_color, arrow_length_ratio=0.05)
    ax.text(*(origin + Z_local * 1.02), 'Z', color=arrow_color)

    direction_global=[0,0,0]
    pos_global=[0,0,0]
    if feature is not None:
       direction_global, pos_global = plot_feature_direction_arrow(ax, dimensions, feature)

    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=30, azim=45)
    ax.set_axis_off()
    plt.tight_layout()

    # Save and show image
    tmpfile = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    plt.savefig(tmpfile.name)
    plt.close(fig)

    display(Image(tmpfile.name))
    time.sleep(0.5)
    return direction_global, pos_global


def plot_feature_direction_arrow(ax, dimensions, feature, length=10.0, color='red'):
    origin_offset = np.array(dimensions["origin_mm"])
    rotation_deg = dimensions["rotation_deg"]
    pos_local = np.array(feature["position_mm"])
    direction_info = feature["machining_direction"]

    # Base direction in user system
    if direction_info["type"] == "main_axis":
        dir_dict = {
            'X': np.array([1.0, 0.0, 0.0]),
            'Y': np.array([0.0, 1.0, 0.0]),
            'Z': np.array([0.0, 0.0, 1.0]),
        }
        direction_local = dir_dict.get(direction_info["reference_axis"].upper(), np.array([0.0, 0.0, 1.0]))
        direction_local *= direction_info.get("direction", 1)

    elif direction_info["type"] == "inclined_direction":
        angle_xy = np.radians(direction_info["angle_xy_deg"])
        angle_z = np.radians(direction_info["angle_z_deg"])
        sign = direction_info.get("direction", 1)

        r_xy = np.cos(angle_z)
        dx = r_xy * np.cos(angle_xy)
        dy = r_xy * np.sin(angle_xy)
        dz = np.sin(angle_z)
        direction_local = np.array([dx, dy, dz])
        direction_local *= sign
        direction_local /= np.linalg.norm(direction_local)

    else:
        raise ValueError("Invalid machining direction format.")

    # Tranfer position and direction in global system
    pos_global, R = transform_local_to_world(pos_local, origin_offset, rotation_deg)
    direction_global = R @ direction_local

    # Draw arrow
    ax.quiver(*pos_global, *(direction_global * length), color=color, arrow_length_ratio=0.1)
    ax.text(*(pos_global + direction_global * length * 1.05), 'Hole', color=color)
    return direction_global, pos_global


# Example gear dimensions
dimensions = {
    "number_of_teeth": 20,
    "outer_diameter": 100.0,           # mm
    "root_diameter": 80.0,             # mm
    "tooth_top_width": 6.0,            # mm
    "tooth_base_width": 10.0,          # mm
    "tooth_height": (100 - 80) / 2,    # mm
    "gear_thickness": 10.0,            # mm
    "tooth_clearance_angle": 0.0,       # degrees (not used in current version)
    "origin_mm": [0, 0, 10],
    "rotation_deg": [0, 0, 0]
}
feature= {
          "diameter_mm": 20,
          "position_mm": [0, 0, 0],
          #"machining_direction":{"type": "inclined_direction", "angle_xy_deg": 0, "angle_z_deg": 70, "sign": -1}
          "machining_direction": {"type": "main_axis",
                "reference_axis": 'Y',
                "direction": 1}
         }
plot_gear_3d(dimensions,feature)

# Text Input

In [ ]:
def collect_part_info():
    SUPPORTED_GEOMETRIES = ["Plate", "Cylinder", "Gear", "Flange", "Bracket", "Block"]
    print("Fill in the following information about the part to be machined:\n")

    material = input("Material: ")
    print("Select the type of geometry:")
    for i, g in enumerate(SUPPORTED_GEOMETRIES):
        print(f"{i+1}. {g}")

    choice = int(input("Enter the corresponding number: ")) - 1
    geometry = SUPPORTED_GEOMETRIES[choice]
    dimensions = ask_dimensions(geometry)
    default_origin, user_origin = define_reference_origin(geometry, dimensions)
    features = collect_features(geometry,dimensions)
    tolerance_val = input("Required tolerance ± ... mm → Enter only the value: ")
    tolerance = f"±{tolerance_val} mm"

    finish_val = input("Surface finish Ra < ... µm → Enter only the value: ")
    finish = f"Ra < {finish_val} µm"
    del dimensions["origin_mm"]
    del dimensions["rotation_deg"]
    part_info = {
        "geometry": geometry,
        "material": material,
        "default_origin": default_origin,
        "geometry_parameters": dimensions,
        "features": features,
        "tolerance": tolerance,
        "surface_finish": finish
    }

    print("\nCollected information:")
    for k, v in part_info.items():
        print(f"- {k}: {v}")

    return part_info

def ask_dimensions(geometry):
    print(f"\nEnter the dimensional parameters for geometry: {geometry}\n")
    dimensions = {}

    if geometry == "Plate":
        dimensions["length_mm"] = float(input("Length (mm): "))
        dimensions["width_mm"] = float(input("Width (mm): "))
        dimensions["thickness_mm"] = float(input("Thickness (mm): "))

    elif geometry == "Cylinder":
        dimensions["diameter_mm"] = float(input("Diameter (mm): "))
        dimensions["height_mm"] = float(input("Height (mm): "))

    elif geometry == "Gear":
        print("\n--- Define gear parameters ---")
        # Core parameters for trapezoidal gear shape
        dimensions["number_of_teeth"] = int(input("Number of teeth: "))
        dimensions["module"] = float(input("Module (mm): "))
        dimensions["pressure_angle"] = float(input("Pressure angle (degrees), typically 20: "))

        # Derived involute geometry
        dimensions["pitch_diameter"] = dimensions["module"] * dimensions["number_of_teeth"]
        dimensions["addendum"] = dimensions["module"]
        dimensions["dedendum"] = 1.25 * dimensions["module"]
        dimensions["outer_diameter"] = dimensions["pitch_diameter"] + 2 * dimensions["addendum"]
        dimensions["root_diameter"] = dimensions["pitch_diameter"] - 2 * dimensions["dedendum"]

        # Useful for trapezoidal plotting
        dimensions["tooth_height"] = dimensions["outer_diameter"] / 2 - dimensions["root_diameter"] / 2
        dimensions["tooth_top_width"] = float(input("Tooth top width (Width of the trapezoid at the tip): "))
        dimensions["tooth_base_width"] = float(input("Tooth base width (Width of the trapezoid at the base): "))

        # 3D extrusion
        dimensions["gear_thickness"] = float(input("Gear thickness (Extrusion depth in mm): "))

        # Optional manufacturing tolerance
        dimensions["tooth_clearance_angle"] = float(input("Tooth clearance angle (deg, optional, set 0 if not needed): "))
    elif geometry == "Flange":
        dimensions["outer_diameter_mm"] = float(input("Outer diameter (mm): "))
        dimensions["thickness_mm"] = float(input("Thickness (mm): "))
        dimensions["number_of_holes"] = int(input("Number of holes: "))

    elif geometry == "Bracket":
        dimensions["width_mm"] = float(input("Width (mm): "))
        dimensions["height_mm"] = float(input("Height (mm): "))
        dimensions["thickness_mm"] = float(input("Thickness (mm): "))
        dimensions["bend_angle_deg"] = float(input("Bend angle (°): "))

    elif geometry == "Block":
        dimensions["length_mm"] = float(input("Length (mm): "))
        dimensions["width_mm"] = float(input("Width (mm): "))
        dimensions["height_mm"] = float(input("Height (mm): "))

    else:
        print("Unrecognized geometry. No parameters required.")

    return dimensions

def describe_transformation(translation_mm, rotation_deg_xyz):
    axes = ['X', 'Y', 'Z']
    description = {}

    # Translations description
    trans_desc = []
    for i, val in enumerate(translation_mm):
        if val != 0:
            direction = '+' if val > 0 else '-'
            trans_desc.append(f"{direction}{abs(val)} mm along {axes[i]}")
    description['translation'] = "No translation" if not trans_desc else "New origin is shifted " + ", ".join(trans_desc) + " from default origin"

    # Rotations description
    rot_desc = []
    planes = ['YZ', 'XZ', 'XY']
    for i, angle in enumerate(rotation_deg_xyz):
        if angle != 0:
            rot_desc.append(f"Rotated {abs(angle)}° counter-clockwise around {axes[i]} axis ({planes[i]} plane)")
    description['rotation'] = "No rotation" if not rot_desc else ", ".join(rot_desc)

    return description

def define_reference_origin(geometry, dimensions):
    print("\n--- DEFINITION OF THE ORIGIN AND AXES OF THE REFERENCE SYSTEM (WCS) ---")
    dimensions["origin_mm"] = [0.0, 0.0, 0.0]
    dimensions["rotation_deg"]=[0.0, 0.0, 0.0]
    plot_reference_system(geometry, dimensions)

    if geometry in ["Plate", "Block"]:
        origin_descr = "Center of the mid-plane"
        orientation_descr = "X along the length direction, Y along the width direction, Z along thickness direction"
    elif geometry in ["Cylinder"]:
        origin_descr = "Center point of the bottom face of the cylinder"
        orientation_descr = "X: radial direction, orthogonal to cylinder axis, Y: radial direction, orthogonal to cylinder axis, Z: aligned with the cylinder axis, pointing upward"
    elif geometry in [ "Gear", "Flange"]:
        origin_descr = "Center of the lower surface of the Gear"
        orientation_descr = "X radial, Y radial, Z along the cylinder axis (upwards)"
    elif geometry == "Bracket":
        origin_descr = "Approximate center of the bent plane"
        orientation_descr = "X along the base, Y along height, Z through thickness"
    else:
        origin_descr = "Estimated geometric center of the part"
        orientation_descr = "X, Y, Z oriented according to neutral geometry (Cartesian)"

    print(f"\nSuggested origin: **{origin_descr}** → considered as (0, 0, 0)")
    print(f"Default axis orientation: {orientation_descr}")

    choice = input("\nDo you want to use this origin and orientation? (y = yes, n = no): ").strip().lower()

    if choice == 'y':
         default_dict = {
            "origin_mm": [0.0, 0.0, 0.0],
            "origin_description": origin_descr,
            "axis_orientation": orientation_descr,
            "coordinate_system": "Cartesian",
            "notes": "Z is assumed to be the spindle/tool axis in machining context"
         }
         return default_dict, default_dict

    elif choice == 'n':
        print("\nEnter an offset from the suggested point (in mm):")
        x = float(input("Offset X: "))
        y = float(input("Offset Y: "))
        z = float(input("Offset Z: "))
        new_origin = [round(x, 2), round(y, 2), round(z, 2)]
        dimensions["origin_mm"] = new_origin
        rotation = [0.0, 0.0, 0.0]
        rot_choice = input("Do you also want to rotate the reference system? (y/n): ").strip().lower()
        if rot_choice == 'y':
            print("Enter rotation angles around the axes (in degrees):")
            rx = float(input("Counter-Clock wise rotation around X in plane YZ (°): "))
            ry = float(input("Counter-Clock wise rotation around Y in plane XZ(°):"))
            rz = float(input("Counter-Clock wise rotation around Z in plane XY(°):"))
            rotation = [round(rx, 2), round(ry, 2), round(rz, 2)]
            dimensions["rotation_deg"] = rotation;
        plot_reference_system(geometry, dimensions)
        description= describe_transformation(new_origin, rotation)
        user_modified_dict = {
            "translation_mm": new_origin,
            "rotation_deg_xyz": rotation,
            "rotation_order": "XYZ",
            "transformation_description": description,
            "reference": "Transform applied to default_origin",

        }
        default_dict = {
            "origin_mm": [0.0, 0.0, 0.0],
            "origin_description": origin_descr,
            "axis_orientation": orientation_descr,
            "coordinate_system": "Cartesian",
            "notes": "Z is assumed to be the spindle/tool axis in machining context"
        }
        return default_dict, user_modified_dict
    else:
        default_dict = {
            "origin_mm": [0.0, 0.0, 0.0],
            "origin_description": origin_descr,
            "axis_orientation": orientation_descr,
            "coordinate_system": "Cartesian",
            "notes": "Z is assumed to be the spindle/tool axis in machining context"
        }
        return default_dict, default_dict

def define_machining_direction():
    print("\nDefinition of machining direction:")
    choice = input("Do you want to align machining to one of the main X/Y/Z axes? (y/n): ").strip().lower()

    if choice == 'y':
        axis = input("Specify the machining axis (X / Y / Z): ").strip().upper()
        sign = input("Is the direction positive? (y/n): ").strip().lower()
        sign = 1 if sign == 'y' else -1
        if axis in ["X", "Y", "Z"]:
            return {
                "type": "main_axis",
                "reference_axis": axis,
                "direction": sign,
            }
        else:
            print("Invalid axis. Using Z axis by default.")
            return {
                "type": "main_axis",
                "reference_axis": axis,
                "direction": 1,
            }


    elif choice == 'n':
        print("Specify the inclined machining direction:")
        angle_xy = float(input("Angle in the XY plane w.r.t. X axis (degrees, clockwise): "))
        angle_z = float(input("Inclination angle w.r.t. Z axis (degrees, clockwise): "))
        sign = input("Is the direction positive (forward)? (y/n): ").strip().lower()
        sign = 1 if sign == 'y' else -1
        return {
            "type": "inclined_direction",
            "angle_xy_deg": round(angle_xy, 2),
            "angle_z_deg": round(angle_z, 2),
            "direction": sign,
        }


    else:
        print("Invalid answer. Using Z+ by default.")
        return {
            "type": "main_axis",
            "reference_axis": "Z",
            "direction": 1
        }


def collect_features(geometry,dimensions):
    print("\n--- GEOMETRIC FEATURE COLLECTION ---")
    features = []
    count = 1

    FEATURE_TYPES = {
        "1": "Through hole",
        "2": "Blind hole",
        "3": "Rectangular pocket",
        "4": "Circular pocket",
        "5": "Slot",
        "6": "Chamfer",
        "7": "Fillet",
        "8": "Hole + Thread",
        "9": "Thread",
        "10": "Other (free description)",

    }

    while True:
        choice = input(f"\nDo you want to add feature #{count}? (y/n): ").strip().lower()
        if choice == 'n':
            break
        elif choice != 'y':
            print("Invalid response. Please enter 'y' or 'n'.")
            continue

        print("\nSelect the type of feature:")
        for code, name in FEATURE_TYPES.items():
            print(f"{code}. {name}")
        ftype = input("Number corresponding to the feature: ").strip()

        if ftype not in FEATURE_TYPES:
            print("Invalid feature type.")
            continue

        feature = {"type": FEATURE_TYPES[ftype]}
        if ftype == "1":  # Through hole
            feature["type"] = "hole"
            feature["subtype"] = "through"
            feature["diameter_mm"] = float(input("Diameter (mm): "))
            print("Specify the **starting point** of the hole (entry point for the tool):")
            x = float(input("Start X (mm): "))
            y = float(input("Start Y (mm): "))
            z = float(input("Start Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["position_reference"] = "default_origin"
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]


        elif ftype == "2":  # Blind hole
            feature["diameter_mm"] = float(input("Diameter (mm): "))
            feature["depth_mm"] = float(input("Depth (mm): "))
            print("Specify the **starting point** of the hole (entry point for the tool):")
            x = float(input("Start X (mm): "))
            y = float(input("Start Y (mm): "))
            z = float(input("Start Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["position_reference"] = "default_origin"
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]


        elif ftype == "3":  # Rectangular pocket
            print("Dimension are specified with respect to user reference system:")
            feature["dimension_x"] = float(input("x (mm): "))
            feature["dimension_y"] = float(input("y (mm): "))
            feature["dimension_z"] = float(input("z (mm): "))
            print("Specify the center of the pocket (entry point for the tool):")
            x = float(input("Center X (mm): "))
            y = float(input("Center Y (mm): "))
            z = float(input("Center Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["position_description"] = "Center of the pocket"
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]


        elif ftype == "4":  # Circular pocket
            feature["diameter_mm"] = float(input("Diameter (mm): "))
            feature["depth_mm"] = float(input("Depth (mm): "))
            print("Specify the center of the pocket (entry point for the tool):")
            x = float(input("Center X (mm): "))
            y = float(input("Center Y (mm): "))
            z = float(input("Center Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["position_description"] = "Center of the pocket"
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]


        elif ftype == "5":  # Slot
            feature["length_mm"] = float(input("Length (mm): "))
            feature["width_mm"] = float(input("Width (mm): "))
            print("Specify the center of the Slot (entry point for the tool):")
            x = float(input("Center X (mm): "))
            y = float(input("Center Y (mm): "))
            z = float(input("Center Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]


        elif ftype in ["6", "7"]:  # Chamfer / Fillet
            feature["radius_mm"] = float(input("Radius (mm): "))
            feature["edge"] = input("Edge to apply the feature (user description required) (e.g., top edge, right edge...): ").strip()

        elif ftype == "8":  # Thread -----------------------------------------------------
            feature["operation_sequence"] = ["drill", "thread"]
            feature["pre_hole_diameter_mm"] = float(
                input("Pre-hole diameter (mm): ")
            )
            feature["pre_hole_depth_mm"] = float(
                input("Pre-hole depth (mm): ")
            )
            #––  Threading parameters ––#
            feature["thread_type"] = input(
                "Thread type ('internal' or 'external'): "
            ).strip()
            feature["standard"] = input(
                "Thread standard (e.g. 'M6', 'M10x1.25', 'UNC 1/4'): "
            ).strip()
            feature["nominal_diameter_mm"] = float(
                input("Nominal diameter (mm): ")
            )
            feature["pitch_mm"] = float(input("Thread pitch (mm): "))
            feature["depth_mm"] = float(
                input("Thread depth (mm): ")
            )
            #––  Machining options ––#
            feature["coolant"] = None
            feature["peck_drilling"] = None

            #––  Positioning ––#
            print("Specify the center of the thread (tool entry point):")
            x = float(input("Center X (mm): "))
            y = float(input("Center Y (mm): "))
            z = float(input("Center Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["position_reference"] = "default_origin"
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
                del feature["machining_direction"]["direction"]
                del feature["machining_direction"]["reference_axis"]



        elif ftype == "9":  # Thread
            feature["thread_type"] = input("Thread type ( type : 'internal' o 'external'): ").strip()
            feature["standard"] = input(" Standard: e.g. 'M6' , ' M10x1.25', 'UNIC 1/4' better if according to the norm: ").strip()
            feature["nominal_diameter_mm"] = float(input("Nominal diameter (mm): "))
            feature["pitch_mm"] = float(input("Pitch (mm): "))
            feature["depth_mm"] = float(input("Depth (mm): "))
            print("Specify the center of the thread (entry point for the tool):")
            x = float(input("Center X (mm): "))
            y = float(input("Center Y (mm): "))
            z = float(input("Center Z (mm): "))
            feature["position_mm"] = [x, y, z]
            feature["machining_direction"] = define_machining_direction()
            direction_global, pos_global =plot_reference_system(geometry, dimensions, feature)
            feature["position_mm"] = pos_global
            feature["position_reference"] = "default_origin"
            sign_val = feature["machining_direction"]["direction"]
            feature["machining_direction"]["direction"] = "positive" if sign_val == 1 else "negative"
            del feature["machining_direction"]["type"]
            feature["machining_direction"]["vector"] = direction_global
            if 'angle_z_deg' in feature["machining_direction"]:
                del feature["machining_direction"]["angle_z_deg"]
                del feature["machining_direction"]["angle_xy_deg"]
            else:
              del feature["machining_direction"]["direction"]
              del feature["machining_direction"]["reference_axis"]

        elif ftype == "10":  # Other
            desc = input("Free description of the feature: ").strip()
            if desc:
                feature["description"] = desc
            else:
                print("Empty description not valid. Feature ignored.")
                continue

        features.append(feature)
        count += 1

    return features

def plot_reference_system(geometry, dimensions,feature=None):
    direction_global=[0,0,0]
    pos_global=[0,0,0]
    if geometry == "Gear":
      if feature is not None:
        direction_global, pos_global=plot_gear_3d(dimensions,feature)
      else:
        plot_gear_3d(dimensions)
    elif geometry == "Cylinder":
      if feature is not None:
        direction_global, pos_global=plot_cylinder_3d(dimensions,feature)
      else:
        plot_cylinder_3d(dimensions)
    elif geometry == "Plate":
      if feature is not None:
        direction_global, pos_global=plot_plate_3d(dimensions,feature)
      else:
        plot_plate_3d(dimensions)
    return direction_global, pos_global


In [ ]:
53# Execute the function to collect input
part_info = collect_part_info()

# Execute the function to collect input
print("-------------- PART DESCRIPTION -----------------------")
for k, v in part_info.items():
        print(f"- {k}: {v}")



# ENVIRONMENT DEFINITION

In [ ]:
def convert_ndarray_to_list(obj):
    if isinstance(obj, dict):
        return {k: convert_ndarray_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_ndarray_to_list(item) for item in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj
# Converti tutto il dizionario in forma serializzabile
serializable_part_info = convert_ndarray_to_list(part_info)

# Ora puoi fare il dump JSON senza errori
json_string = json.dumps(serializable_part_info, indent=2)
print(json_string)
part_info = json_string

 **Environment and prompt setup**

In [ ]:
part_info = {
    "geometry": "Cylinder",
    "material": "steel",
    "default_origin": {
        "origin_mm": [0.0, 0.0, 0.0],
        "origin_description": "Center of base of cylinder",
        "axis_orientation": "X: radial, Y: radial, Z: axial upward",
        "coordinate_system": "Cartesian"
    },
    "geometry_parameters": {
        "diameter_mm": 100.0,
        "height_mm": 180.0
    },
    "features": [
        {
            "type": "Hole + Thread",
            "operation_sequence": ["drill", "thread"],
            "pre_hole_diameter_mm": 4.0,
            "pre_hole_depth_mm": 180.0,
            "thread_type": "internal",
            "standard": "M6",
            "nominal_diameter_mm": 6.0,
            "pitch_mm": 1.0,
            "depth_mm": 40.0,
            "position_mm": [0.0, 0.0, 0.0],
            "machining_direction": {"vector": [0.0, 0.0, 1.0]}
        },
        {
            "type": "Blind hole",
            "diameter_mm": 6.0,
            "depth_mm": 50.0,
            "position_mm": [30.0, -10.0, 0.0],
            "machining_direction": {"vector": [0.0, 0.0, 1.0]}
        },
        {
            "type": "Chamfer",
            "angle_deg": 45,
            "width_mm": 2.0,
            "position_mm": [0.0, 0.0, 180.0],
            "machining_direction": {"vector": [0.0, 0.0, -1.0]}
        },
        {
            "type": "Rectangular pocket",
            "dimension_x": 15.0,
            "dimension_y": 8.0,
            "dimension_z": 5.0,
            "position_mm": [-30.0, 0.0, 120.0],
            "machining_direction": {"vector": [-1.0, 0.0, 0.0]}
        },
        {
            "type": "Slot",
            "width_mm": 3.0,
            "length_mm": 25.0,
            "depth_mm": 4.0,
            "position_mm": [0.0, 35.0, 160.0],
            "machining_direction": {"vector": [-1.0, 0.0, 0.0]}
        }
    ]
}
def convert_ndarray_to_list(obj):
    if isinstance(obj, dict):
        return {k: convert_ndarray_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_ndarray_to_list(item) for item in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj
# Converti tutto il dizionario in forma serializzabile
serializable_part_info = convert_ndarray_to_list(part_info)

# Ora puoi fare il dump JSON senza errori
json_string = json.dumps(serializable_part_info, indent=2)
print(json_string)
part_info = json_string

In [ ]:
part_info = (
    "You are looking at a rectangular aluminium cover plate for a small gearbox. "
    "The finished plate is 120 mm wide (X), 90 mm tall (Y) and 20 mm thick (Z). "
    "Its datum is the exact centre of the top face. "
    "All edges of the top face are destined for a light-cut 1 mm × 45° chamfer after other machining is complete. "
    "The top face needs a 0.8 µm Ra cosmetic finish, so it will be face-milled, as will the bottom face after the part is flipped to reach final thickness. "

    "Features, working from the centre outward: "
    "• Central bearing seat – one blind hole, pilot-drilled to 10 mm, then bored to 30 mm Ø, 15 mm deep, true position ±0.02 mm, surface finish Ra 0.8 µm. "
    "• Alignment dowel hole – one precision reamed hole, 12 mm Ø, 20 mm deep, located 40 mm above the datum on the Y-axis, tolerance H7. "
    "• Threaded service port – one blind tapped hole M10 × 1.5, drill Ø 8.5 mm, 18 mm deep, positioned 35 mm below the datum on the Y-axis. "
    "• Rectangular relief pocket – 70 mm (X) by 50 mm (Y), 5 mm deep, centred on the datum, corner radii 3 mm. "
    "• Oil-feed slot – 60 mm long, 10 mm wide and 10 mm deep, centred on the datum in X and offset +30 mm in Y, running parallel to the Y-axis. "
    "• Four corner mounting holes – through-holes 8 mm Ø, arrayed symmetrically at X ±55 mm, Y ±40 mm; depth equals plate thickness (20 mm)."
)


In [ ]:
part_info = (
    "The component is a square steel mounting base for a robotic arm. "
    "Finished overall size: 200 mm in X, 200 mm in Y, and 25 mm in Z. "
    "The global datum is the intersection of the plate centre and the top surface (Z-positive normal). "

    "General requirements: the top face must meet a surface finish of Ra 1.6 µm, and a 1.5 mm × 45° chamfer is required around every exposed edge once all other geometry is complete. "

    "Features, referenced to the datum (X right, Y up): "

    "• Central pocket — a circular recess 80 mm in diameter, 8 mm deep, concentric with the datum. "

    "• Four M12 threaded inserts — blind holes positioned at X ±65 mm, Y ±65 mm, hole diameter 10.2 mm, depth 22 mm. "

    "• Six through-holes for dowel pins — diameter 6 mm, depth equals plate thickness, centred on a 120 mm-diameter bolt circle measured from the datum; equally spaced every 60°. "

    "• Two cable-pass channels — rectangular slots 15 mm wide, 6 mm deep, length 120 mm, each centred on X ±40 mm and running parallel to the Y-axis (slot ends are semicircular with radius 7.5 mm). "

     "• Identification recess — shallow rectangle 60 mm (X) × 20 mm (Y) × 0.5 mm deep located at X –60 mm, Y +80 mm on the top face. "

     "• Eight counterbored mounting holes on the bottom face — overall diameter 22 mm, counterbore depth 5 mm, clearance hole 13 mm; arranged in two rows along Y ±75 mm with X positions at –80, –40, +40, +80 mm. "

    "• Bottom face must maintain flatness within 0.05 mm over the entire 200 mm × 200 mm area. "
)


In [ ]:
part_info = {
  "geometry": "Plate",
  "material": "copper",
  "default_origin": {
    "origin_mm": [
      0.0,
      0.0,
      0.0
    ],
    "origin_description": "Center of the mid-plane",
    "axis_orientation": "X along the length direction, Y along the width direction, Z along thickness direction",
    "coordinate_system": "Cartesian",
    "notes": "Z is assumed to be the spindle/tool axis in machining context"
  },
  "geometry_parameters": {
    "length_mm": 120.0,
    "width_mm": 120.0,
    "thickness_mm": 20.0
  },
  "features": [
          {
      "type": "Blind hole",
      "diameter_mm": 3.0,
      "depth_mm": 5.0,
      "position_mm": [
        35.0,
        5.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
              {
      "type": "Blind hole",
      "diameter_mm": 3.0,
      "depth_mm": 5.0,
      "position_mm": [
        35.0,
        -5.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
        {
      "type": "Blind hole",
      "diameter_mm": 3.0,
      "depth_mm": 5.0,
      "position_mm": [
        30.0,
        0.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
        {
      "type": "Blind hole",
      "diameter_mm": 3.0,
      "depth_mm": 5.0,
      "position_mm": [
        40.0,
        5.00,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },

    {
      "type": "hole",
      "subtype": "through",
      "diameter_mm": 10.0,
      "position_mm": [
        35.0,
        35.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
    {
      "type": "hole",
      "subtype": "through",
      "diameter_mm": 10.0,
      "position_mm": [
        35.0,
        -35.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
    {
      "type": "hole",
      "subtype": "through",
      "diameter_mm": 10.0,
      "position_mm": [
        -35.0,
        35.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
    {
      "type": "hole",
      "subtype": "through",
      "diameter_mm": 10.0,
      "position_mm": [
        -35.0,
        -35.0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
    {
      "type": "hole",
      "subtype": "through",
      "diameter_mm": 5.0,
      "position_mm": [
        0,
        0,
        20.0
      ],
      "position_reference": "default_origin",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      }
    },
    {
      "type": "Rectangular pocket",
      "dimension_x": 20.0,
      "dimension_y": 20.0,
      "dimension_z": 10.0,
      "position_mm": [
        0.0,
        0.0,
        20.0
      ],
      "position_description": "Center of the pocket",
      "machining_direction": {
        "vector": [
          0.0,
          0.0,
          -1.0
        ]
      },
      "position_reference": "default_origin"
    }
  ],
  "tolerance": "\u00b10.1 mm",
  "surface_finish": "Ra < 5 \u00b5m"
}
def convert_ndarray_to_list(obj):
    if isinstance(obj, dict):
        return {k: convert_ndarray_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_ndarray_to_list(item) for item in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj
# Converti tutto il dizionario in forma serializzabile
serializable_part_info = convert_ndarray_to_list(part_info)

# Ora puoi fare il dump JSON senza errori
json_string = json.dumps(serializable_part_info, indent=2)
print(json_string)
part_info = json_string

In [ ]:
#-------------------- LLM --------------------------------------------------
from google.colab import userdata
openai_api_key = userdata.get('OpenAI_API')
openai.api_key = openai_api_key
ENGINE =   "gpt-4o-mini"

# check if it's a string
TEXT_MODE = isinstance(part_info, str)

# Call API
def call_api(message,temp):
    try:
        response = openai.chat.completions.create(
            model=ENGINE,
            messages=[{"role": "user", "content": message}],
            temperature=temp,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API call error: {e}")
        return None

def clean_llm_json_response(text: str) -> str:
    """
    Cleans the LLM output by removing ```json e ``` delimiters
    Makes the text compatible with json.loads()
    """
    lines = text.strip().splitlines()
    # Remove all the lines that start with ```
    lines = [line for line in lines if not line.strip().startswith("```")]
    return "\n".join(lines).strip()

def token_check(propmt):
  encoding = tiktoken.encoding_for_model(ENGINE)
  num_tokens = len(encoding.encode(propmt))
  print(f"Number of tokens: {num_tokens}")


# ------------------------------ OPERATIONS -------------------------------------
operation_constraints = {
    "Drilling (Through Hole)": {
        "rpm_max": 6000,
        "max_depth": 100,
        "max_tool_diameter": 25,
        "note": "High torque required for deep holes"
    },
    "Drilling (Blind Hole)": {
        "rpm_max": 6000,
        "max_depth": 80,
        "max_tool_diameter": 20,
        "note": "Careful chip evacuation needed"
    },
    "Face Milling": {
        "rpm_max": 8000,
        "max_depth": 5,
        "max_tool_diameter": 50,
        "note": "Large diameter end mills"
    },
    "Slot Milling": {
        "rpm_max": 7000,
        "max_depth": 20,
        "max_tool_diameter": 16,
        "note": "Requires stable fixturing"
    },
    "Contour Milling": {
        "rpm_max": 7500,
        "max_depth": 10,
        "max_tool_diameter": 20,
        "note": "High precision finish"
    },
    "Threading": {
        "rpm_max": 3000,
        "max_feed": 300,
        "max_tool_diameter": 12,
        "note": "Use correct tap cycle"
    },
    "Boring": {
        "rpm_max": 5000,
        "max_depth": 120,
        "max_tool_diameter": 40,
        "note": "Requires boring head attachment"
    },
    "Reaming": {
        "rpm_max": 4000,
        "max_depth": 50,
        "max_tool_diameter": 18,
        "note": "Use coolant for finish"
    },
    "Chamfering": {
        "rpm_max": 10000,
        "max_depth": 2,
        "max_tool_diameter": 10,
        "note": "45-degree chamfer tools"
    },
    "Tapping": {
        "rpm_max": 2500,
        "max_depth": 30,
        "max_tool_diameter": 12,
        "note": "Synchronous spindle required"
    },
    "Pocket Milling": {
    "rpm_max": 7500,
    "max_depth": 20,
    "max_tool_diameter": 20,
    "note": "Use end mills to machine enclosed pockets"
}
}

# ----------------------------------- TOOLS ---------------------------------------------------
available_tools = [
    {"name": "Twist Drill", "diameter_mm": list(range(2, 26))},
    {"name": "End Mill", "diameter_mm": list(range(1, 51))},
    {"name": "Chamfer Mill", "diameter_mm": [5, 10, 15, 20]},
    {"name": "Ball Nose End Mill", "diameter_mm": [3, 6, 12, 16]},
    {"name": "Tapered End Mill", "diameter_mm": [5, 10, 20]},
    {"name": "Reamer", "diameter_mm": [6, 8, 10, 12, 16]},
    {"name": "Boring Head", "diameter_mm": [20, 30, 40]},
    {"name": "Thread Tap", "diameter_mm": [3, 4, 5, 6, 8, 10, 12]},
    {"name": "Countersink", "diameter_mm": [8, 10, 12]},
    {"name": "Face Mill Cutter", "diameter_mm": [50, 63, 80]}
]

#------------- OPERATIONS TO TOOLS ----------------------------------------------------------
operation_to_tool_type = {
    "Drilling (Through Hole)": {"Twist Drill"},
    "Drilling (Blind Hole)": {"Twist Drill"},
    "Face Milling": {"Face Mill Cutter", "End Mill"},
    "Slot Milling": {"End Mill", "Ball Nose End Mill"},
    "Contour Milling": {"End Mill", "Ball Nose End Mill", "Tapered End Mill"},
    "Pocket Milling": {"End Mill", "Ball Nose End Mill"},
    "Chamfering": {"Chamfer Mill", "Countersink"},
    "Tapping": {"Thread Tap"},
    "Threading": {"Thread Tap"},
    "Boring": {"Boring Head"},
    "Reaming": {"Reamer"},
}

# ------------------------- PREFERENCE RULES -----------------------------------
precedence_rules = [
# -------------------------------------
    # HOLES: Rough → Finish sequences
    # -------------------------------------
    ("Drilling",  "Boring"),        # Pilot hole before enlarging bore
    ("Drilling",  "Reaming"),       # Drill hole first, then ream for precise diameter
    ("Drilling",  "Tapping"),       # Drill hole before tapping threads
    ("Drilling",  "Threading"),     # Drill before threading (single-point)
    ("Boring",    "Reaming"),       # Bore hole, then ream precisely

    # -------------------------------------
    # PLANAR MACHINING: Rough → Finish
    # -------------------------------------
    ("Face Milling", "Pocket Milling"),    # Flatten surface before pocket machining
    ("Face Milling", "Slot Milling"),      # Flatten surface before slot machining
    ("Face Milling", "Contour Milling"),   # Flatten surface before contour
    ("Face Milling", "Chamfering"),        # Flatten surface before chamfering

    ("Pocket Milling", "Slot Milling"),    # Pocket before slots if overlapping
    ("Pocket Milling", "Chamfering"),      # Pocket before chamfering edges
    ("Pocket Milling", "Contour Milling"), # Pocket first, then finishing contour

    ("Slot Milling", "Chamfering"),        # Slot before chamfering slot edges

    ("Contour Milling", "Chamfering"),     # Finish contour before chamfering edges

    # -------------------------------------
    # DRILL BEFORE MILL WHEN OVERLAPPING
    # -------------------------------------
    ("Drilling", "Pocket Milling"),        # Drill holes before machining pockets if overlapping
    ("Drilling", "Slot Milling"),          # Drill holes before machining slots if overlapping

    # -------------------------------------
    # ALWAYS-LAST OPERATION
    # -------------------------------------
    ("Any", "Chamfering"),                 # Chamfering generally last
]



# ---  rules text from precedence_rules ---
rules_text = "\n".join(
    f"   - {A} must precede {B}." if A != "Any" else f"   - {B} must always be the last operation."
    for A, B in precedence_rules
)


# ---------------- CATEGORY TO OPERATIONS ----------------------------


def op_category(op_name: str) -> str:
    """
    Converts full operation label to a category keyword
    used in the precedence rules above.
    """
    if "Drilling" in op_name:   return "Drilling"
    if "Boring" in op_name:     return "Boring"
    if "Reaming" in op_name:    return "Reaming"
    if "Tapping" in op_name:    return "Tapping"
    if "Threading" in op_name:  return "Threading"
    if "Face Milling" in op_name:    return "Face Milling"
    if "Slot Milling" in op_name:    return "Slot Milling"
    if "Pocket Milling" in op_name:  return "Pocket Milling"
    if "Contour Milling" in op_name: return "Contour Milling"
    if "Chamfering" in op_name:      return "Chamfering"
    # fallback: return the name itself
    return op_name




# ----------------------- PROMPTING --------------------------------------------------------------

# ---------------------------------------- 1 -----------------------------------------------------
system_message1 = (
    "You are a CNC expert. Your job is to analyse a part description and identify the required machining operations.\n"
    "The part description may arrive in **either** of two formats:\n"
    "  • A valid JSON object (preferred)\n"
    "  • Free text in plain English\n"
    "Parse whichever form is provided and continue with the same workflow. If free text is supplied, silently extract the necessary structured data before proceeding.\n\n"

    "You must:\n"
    "1. List ONLY the operations needed to manufacture the part.\n"
    "2. For each operation, select ONE suitable tool from the available-tools list, choose a diameter **only** from the allowed list for that tool, use only integer diameter.\n"
    "3. Obey all machine constraints: allowed operations, tools, max-RPM, max-depth, max tool Ø.\n"
    "4. If 'coolant' or 'peck_drilling' are None, decide whether to activate them based on depth, material, tool type, "
    "and best practice for chip evacuation/tool life and notw your decision in Notes.\n"
    "5. If a single pass would exceed ANY limit (RPM, depth-per-pass, tool Ø, machine travel, etc.), "
    "SPLIT the task into as many consecutive passes as needed. "
    "Each pass must respect the constraints and appear as a separate entry in the JSON array, clearly indicating pass # "
    "in the Notes (e.g. 'Pass 1/3') and justify briefly the decision, include the machine direction in the note like 'dir: -Z'.\n\n"

    "Important formatting rule:\n"
    "- When specifying the tool, ALWAYS use the format: \"Tool Name, diameter: N\"\n"
    "- Do NOT use symbols like Ø, ⌀, D, or 'diam.'\n"
    "- Examples: \"End Mill, diameter: 10\", \"Twist Drill, diameter: 5\"\n\n"

    "Position field rules"
    "If the part description is **JSON**, give either"
    "– a numeric dict  → {'X': …, 'Y': …, 'Z': …}  (mm)"
    "• If the part description is **free text in which the part is described by sentence in egnlish**, you may use a concise descriptive label"
    "(e.g. 'Top Face', 'located at X –60 mm, Y +80 mm on the top face') BUT ONLY IN THIS CASE DO NOT USE concise descriptive label for JSON."
    "Any other wording is invalid."

    "Available operations:\n"
    + "\n".join(
        f"- {op}: {c['note']} (RPM ≤{c['rpm_max']}, Depth ≤{c.get('max_depth','N/A')}, "
        f"Tool Ø ≤{c.get('max_tool_diameter','N/A')})"
        for op, c in operation_constraints.items()
    )
    + "\n\nAvailable tools:\n"
    + "\n".join(f"- {t['name']}, diameters: {t['diameter_mm']}" for t in available_tools)
    + "\n\nRespond ONLY in valid JSON with EXACTLY this structure:\n"
    """{
      "Operations": [
        {
          "Operation": "...",
          "Tool": "...",
          "Position": "...",
          "RPM": "...",
          "Depth": "...",
          "Coolant": "...",
          "Peck_Drilling": "...",
          "Notes": "..."
        }
      ]
    }"""
)


def build_prompt1(data):
    """Embed the part description, handling JSON dicts/lists **or** raw text."""
    if isinstance(data, (dict, list)):
        description = json.dumps(data, indent=2)
    else:
        description = str(data)
    return f"\nHere is the part description:\n{description}"
# --------------------------------- 2 --------------------------------------------------
system_message2 = (
    "You are a CNC process planner.\n\n"

    "Reference frame:\n"
    "- Origin (0,0,0) is the centre of the machine table, Z=0 plane is the bed surface.\n"
    "- +Z is the spindle/tool direction; machining in −Z collides with the bed.\n\n"
    " Hard collision rule → No drilling, tapping, pocketing or boring may start at Z ≤ 0 **unless** the part is FIRST elevated (e.g. on parallels) or flipped so the face is above the bed"
    "**Goal**: Produce the most efficient machining plan that minimises (1) total setups, and (2) total tool changes, while remaining physically feasible.\n\n"

    "**Setup planning**\n"
    "- For each setup give: Setup number, Orientation (plain English), Fixturing, Notes.\n"
    "- Avoid machining a face or edge that is in contact with the machine bed unless you explicitly flip/elevate the part.\n"
    "- Chamfering operations must flipped, or moved to another setup if the target edge belongs to a face resting on the bed or clamped.\n"
    "- Chamfering operations must be flipped, or moved to another setup if the target edge belongs to a face resting on the bed or clamped.\n"
    "  If chamfering affects multiple faces in different orientations, you must define a separate chamfering step for each affected face, assigned to the correct setup.\n"
    "- If several features can be reached with the same orientation, keep them in the **same setup**.\n\n"

    "**Process planning**\n"
    "- Assign every operation to a valid setup according to its machining direction.\n"
    "- **Ordering rule**:\n"
    "  1. Group consecutive operations by *tool* (same cutter / drill / tap) to avoid unnecessary tool changes.\n"
    "  2. Within the same tool, group by *setup* so the spindle stays in that setup until all operations for that tool are done.\n"
    "- Respect precedence (e.g. drill → ream → tap, rough → finish).\n"
    "- Provide RPM, Feed and Depth that respect the machine limits and material.\n\n"

    "**Hard constraints**\n"
    "- Do not assign an operation to an orientation where the tool would collide with the table or clamps.\n"
    "- Do not add commentary or markdown in the output.\n\n"


    "Important formatting rule:\n"
    "If the part description is **JSON**, give either"
    "– a numeric dict  → {'X': …, 'Y': …, 'Z': …}  (mm) **or** "
    "– the exact feature ID from the JSON (e.g. 'F2')."
    "• If the part description is **free text**, you may use a concise descriptive label"
    "(e.g. 'Top Face', 'Central bearing seat')."
    "Any other wording is invalid."

    "**Return ONLY valid JSON in exactly this structure**:\n"
     """{
      "Setup plan": [
        {
          "Setup number": 1,
          "Orientation": "...",
          "Fixturing": "...",
          "Notes": "..."
        }
      ],
      "Process plan": [
        {
          "Step number": 1,
          "Setup number": 1,
          "Operation": "...",
          "Tool": "...",
          "Position": "...",
          "RPM": ...,
          "Feed": ...,
          "Depth": ...,
          "Notes": "..."
        }
      ]
    }"""
)


def build_prompt2(part_info, operations_output):
    return (
        f"Part description:\n{json.dumps(part_info, indent=2)}\n\n"
        f"Operations and selected tools:\n{json.dumps(operations_output['Operations'], indent=2)}"
    )


# ----------------------------------------------- 3 --------------------------------------------
system_message3 = (
    "You are a CNC process **validator and final optimiser**.\n\n"
    "Reference frame: origin at bed centre, Z=0 at bed surface, +Z upwards.\n\n"

    "** Do NOT change tools or tools diameters**"

    "**Validate feasibility**\n"
    "- For every operation, confirm that its machining direction is allowed by its assigned setup orientation.\n"
    "- Example rule: drilling in Z is impossible if the drilled face is resting on the table.\n"
    "- Add / edit setups only if strictly necessary for physical access.\n\n"

    "**Re-optimise**\n"
    "- After all fixes, re-order the Process plan again so that:\n"
    "  1. All operations using the same tool are contiguous.\n"
    "  2. The total number of setups is the minimum still required.\n"
    "- Keep precedence constraints intact.\n\n"

    "**Checks**\n"
    "- No redundant tool change (a tool should not be re-mounted later if it could have been kept).\n"
    "- Depths, RPM, feeds must stay within provided machine limits.\n\n"
    "- No face or edge may be machined while clamped against the bed.\n"
    "- Chamfering operations must be flipped, or moved to another setup if the target edge belongs to a face resting on the bed.\n"


    "- If any step violates the collision rule, flip it **or** elevate the part and reorder to keep tool grouping minimal  "

    "**Output requirements**\n"
    "- Return **only** valid JSON (same schema as before).  \n"
    "- Absolutely no explanations, comments, or markdown outside the JSON."
)




def build_prompt3(part_info, process_plan_dict):
    return (
        f"Part description:\n{json.dumps(part_info, indent=2)}\n\n"
        f"Current plan:\n{json.dumps(process_plan_dict, indent=2)}"
    )

# ------------------------ ERROR MESSAGE ---------------------------------------
system_message4 = (
    "You are a CNC process diagnostics assistant.\n"
    "Your task is to analyze a failing process plan.\n\n"
    "The plan failed validation. You are given:\n"
    "- The last generated plan (which failed)\n"
    "- The list of validation errors\n"
    "- The general machine constraints\n\n"
    "Your goal is to understand what **could be wrong** with the user input, geometry, or constraints "
    "that might have caused this failure. Then suggest potential changes that could help fix it.\n\n"
    "Do NOT attempt to regenerate a valid plan. Just explain and advise.\n"
)

# VALIDATION

In [ ]:

# ----------- PROCEDURE ORDER VALIDATION -------------------------------------
def validate_precedence(process_plan: List[Dict]) -> Tuple[bool, str]:
    """
    Checks precedence rules within each Setup number, but only applies rules
    for operations that are both present in the current setup.
    Returns (is_valid, error_string)
    """
    from collections import defaultdict

    # Group steps by Setup number
    steps_by_setup = defaultdict(list)
    for step in process_plan:
        steps_by_setup[step["Setup number"]].append(step)

    errors = []

    for setup_no, steps in steps_by_setup.items():
        # Order steps by Step number
        steps_sorted = sorted(steps, key=lambda s: s["Step number"])

        # Extract operation categories actually present in this setup
        present_categories = [op_category(s["Operation"]) for s in steps_sorted]
        present_set = set(present_categories)

        seen = set()

        for step in steps_sorted:
            idx = step["Step number"]
            cat = op_category(step["Operation"])

            for A, B in precedence_rules:
                if B != "Any" and B != cat:
                    continue  # this rule doesn't apply to this step

                if A != "Any" and (A not in present_set or B not in present_set):
                    continue  # skip rules if A or B not in current setup

                # Rule applies, check it
                if A != "Any" and A not in seen:
                    errors.append(
                        f"Setup {setup_no}, Step {idx}: '{cat}' appears before any '{A}'."
                    )

                if A == "Any" and cat == "Chamfering":
                    if idx != steps_sorted[-1]["Step number"]:
                        errors.append(
                            f"Setup {setup_no}, Step {idx}: Chamfering should be last in its setup."
                        )

            # Mark this category as seen
            seen.add(cat)

    if errors:
        return False, "\n".join(errors)

    return True, ""


BOOL_STR = {"yes","no","true","false","on","off"}
def _boolish(val):                              # accepts True/False or Yes/No strings
    return isinstance(val, bool) or (isinstance(val, str) and val.lower() in BOOL_STR)


# -------------- VALIDATION CALL 1 -------------------------------------------------
def validate_call1(response_1: str) -> Tuple[bool, str]:
    import json, re
    from collections import defaultdict

    try:
        data = json.loads(response_1)
        operations = data.get("Operations", [])
    except json.JSONDecodeError as e:
        return False, f"JSON parsing error: {e}"

    errors = []

    # Valid operations from constraints
    valid_operations = set(operation_constraints.keys())

    # Tool dictionary: tool name → set of available diameters
    tool_dict = {tool["name"]: set(tool["diameter_mm"]) for tool in available_tools}

    # Operation → allowed tool types
    # (assumes operation_to_tool_type exists)
    global operation_to_tool_type

    depth_accum = defaultdict(float)
    depth_target = defaultdict(float)

    for idx, op in enumerate(operations, 1):
        op_name = op.get("Operation", "").strip()
        tool_str = op.get("Tool", "").strip()

        # ----- Check 1: Operation exists -----
        if op_name not in valid_operations:
            errors.append(f"Step #{idx}: '{op_name}' is not a supported operation.")
            continue

        # Get constraints for this operation
        limits = operation_constraints.get(op_name, {})

        # ----- Check 2: Tool format -----
        match = re.match(r"(.+?),\s*diameter[:\s]*([0-9]+)", tool_str)
        if not match:
            errors.append(f"Step #{idx}: Tool format invalid → '{tool_str}' (expected 'Tool Name, diameter: N')")
            continue

        tool_name = match.group(1).strip()
        try:
            tool_diameter = float(match.group(2))
        except ValueError:
            errors.append(f"Step #{idx}: Tool diameter is not a valid integer → '{tool_str}'")
            continue

        # ----- Check 3: Tool exists -----
        if tool_name not in tool_dict:
            errors.append(f"Step #{idx}: Tool '{tool_name}' is not in available tool list.")
            continue

        # ----- Check 4: Diameter is allowed -----
        if tool_diameter not in tool_dict[tool_name]:
            valid_sizes = ", ".join(map(str, sorted(tool_dict[tool_name])))
            errors.append(f"Step #{idx}: Tool '{tool_name}' does not support Ø{tool_diameter}. Valid: [{valid_sizes}]")

        # ----- Check 5: Tool type matches operation type -----
        valid_tools_for_op = operation_to_tool_type.get(op_name, set())
        if tool_name not in valid_tools_for_op:
            valid_tool_list = ", ".join(valid_tools_for_op) if valid_tools_for_op else "N/A"
            errors.append(
                f"Step #{idx}: Tool '{tool_name}' is not valid for operation '{op_name}'. Expected one of: [{valid_tool_list}]"
            )

        # ----- numeric fields -----
        try:
            depth_val = float(op.get("Depth", 0))
        except Exception:
            errors.append(f"Step #{idx}: Invalid depth value → '{op.get('Depth')}'")
            continue

        max_pass_depth = limits.get("max_depth")
        if depth_val and max_pass_depth and depth_val > max_pass_depth:
            errors.append(f"Step #{idx}: Depth {depth_val} exceeds single-pass limit ({max_pass_depth}).")

        # Accumulate to verify depth completeness
        key = (op_name.lower(), op.get("Position", ""), tool_name)
        if depth_val:
            depth_accum[key]  += depth_val
            depth_target[key] = max(depth_target[key], depth_val)

            if max_pass_depth and depth_val > max_pass_depth:
                if "pass" not in op.get("Notes", "").lower():
                    errors.append(f"Step #{idx}: Multi-pass op needs 'Pass N/M' in Notes.")

    # Check completeness for multi-pass ops (if depth split across multiple steps)
    for key, total in depth_accum.items():
        op_name, pos_str, tool_name = key
        pass_limit = operation_constraints.get(op_name.title(), {}).get("max_depth")
        if not pass_limit:
            continue  # No limit defined → skip check
        if depth_target[key] > pass_limit and abs(total - depth_target[key]) > 1e-3:
            errors.append(
                f"Multi-pass incomplete for '{op_name}' at {pos_str}: "
                f"requested {depth_target[key]} mm, cumulative passes {total:.1f} mm."
            )

    # ----- Return result -----
    if errors:
        error_msg = "\n".join(errors)
        return False, error_msg

    return True, ""


# ------------------- VALIDATE CALL 2 ----------------------------------------------

def validate_call2(response_2: str) -> Tuple[bool, str]:
    try:
        data = json.loads(response_2)
        process_plan = data.get("Process plan", [])
    except json.JSONDecodeError as e:
        return False, f"JSON parsing error: {e}"

    errors = []

    for step in process_plan:
        step_num = step.get("Step number", "?")
        operation = step.get("Operation", "").strip()
        rpm = step.get("RPM", "")
        depth = step.get("Depth", "")
        tool_str = step.get("Tool", "").strip()

        # --------- Check 1: operation must exist ---------
        if operation not in operation_constraints:
            errors.append(f"Step #{step_num}: Unknown operation '{operation}'")
            continue  # Skip parameter checks

        op_constraints = operation_constraints[operation]

        # --------- Check 2: RPM limit ---------
        if "rpm_max" in op_constraints:
            try:
                rpm_val = float(rpm)
                if rpm_val > op_constraints["rpm_max"]:
                    errors.append(
                        f"Step #{step_num}: RPM {rpm_val} exceeds max RPM ({op_constraints['rpm_max']}) for '{operation}'"
                    )
            except ValueError:
                errors.append(f"Step #{step_num}: RPM '{rpm}' is not a valid number")

        # --------- Check 3: Depth limit ---------
        if "max_depth" in op_constraints:
            try:
                depth_val = float(depth)
                if depth_val > op_constraints["max_depth"]:
                    errors.append(
                        f"Step #{step_num}: Depth {depth_val} exceeds max depth ({op_constraints['max_depth']}) for '{operation}'"
                    )
            except ValueError:
                errors.append(f"Step #{step_num}: Depth '{depth}' is not a valid number")

        # --------- Check 4: Tool diameter limit ---------
        if "max_tool_diameter" in op_constraints:
            match = re.search(r"diameter[:\s]*([0-9]+)", tool_str)
            if match:
                try:
                    tool_diameter = int(match.group(1))
                    if tool_diameter > op_constraints["max_tool_diameter"]:
                        errors.append(
                            f"Step #{step_num}: Tool Ø{tool_diameter} exceeds max Ø{op_constraints['max_tool_diameter']} for '{operation}'"
                        )
                except ValueError:
                    errors.append(f"Step #{step_num}: Invalid tool diameter in '{tool_str}'")
            else:
                errors.append(f"Step #{step_num}: Tool diameter not found in '{tool_str}'")

    # --------- Check Order Consistency ---------
    seq_ok, seq_errors = validate_precedence(process_plan)
    if not seq_ok:
        errors.append(seq_errors)

    if errors:
        return False, "\n".join(errors)

    return True, ""


# ----------------------------- VALIDATE CALL 3 ---------------------------------

def validate_call3(response_3, tol_xy=0.05):
    """Return (ok:bool, message:str)
    Performs three layers of checks:
      1. JSON integrity & position parsing
      2. *Feasibility* of each step w.r.t. its setup orientation
      3. *Precedence* rules – both global (validate_precedence) and local (drilling→threading per feature)
      4. Warns for redundant tool changes
    tol_xy – tolerance (mm) for matching XY positions when pairing drilling & threading.
    """
    import json, re

    # ---- helpers ---------------------------------------------------------
    def _parse_vec(s):
        if isinstance(s, (list, tuple)):
            return tuple(map(float, s[:3]))
        nums = re.findall(r"[-+]?\d*\.?\d+", s)
        if len(nums) < 3:
            raise ValueError("Need 3 coords")
        return tuple(map(float, nums[:3]))

    def _same_xy(p1, p2):
        return abs(p1[0]-p2[0]) <= tol_xy and abs(p1[1]-p2[1]) <= tol_xy

    # ---- load JSON -------------------------------------------------------
    try:
        data = json.loads(response_3)
        setups = {s["Setup number"]: s for s in data["Setup plan"]}
        steps  = data["Process plan"]
    except Exception as e:
        return False, f"JSON error: {e}"

    errors = []

    # ---- build drilling index -------------------------------------------
    drilled_xy = []
    for st in steps:
        op = st.get("Operation", "").lower()
        try:
            xyz = _parse_vec(st.get("Position", "[0,0,0]"))
        except Exception:
            continue
        if any(key in op for key in ("drill", "hole", "boring")):
            drilled_xy.append((xyz[0], xyz[1]))

    # ---- step‑by‑step checks -------------------------------------------
    prev_tool = None
    for st in steps:
        sn   = st["Step number"]
        op_l = st.get("Operation", "").lower()
        try:
            x,y,z = _parse_vec(st.get("Position", "[0,0,0]"))
        except Exception:
            errors.append(f"Step {sn}: Invalid Position format")
            continue
        setup_no = st.get("Setup number")
        orientation = setups.get(setup_no, {}).get("Orientation", "").lower()

        # local precedence: threading after drilling same XY
        if "thread" in op_l and not any(_same_xy((x,y), (dx,dy)) for dx,dy in drilled_xy):
            errors.append(f"Step {sn}: Threading before drilling for the same hole")

        # simple feasibility rules
        if "vertical" in orientation and z <= 0 and "drill" in op_l:
            errors.append(f"Step {sn}: Drilling downward into machine bed in vertical setup")
        if "flipped" in orientation and z >= 0 and "drill" in op_l:
            errors.append(f"Step {sn}: Drilling upward into clamps in flipped setup")
        if "horizontal" in orientation and "pocket" in op_l and abs(z) < 1e-3:
            errors.append(f"Step {sn}: Pocket milling at Z=0 conflicts with bed in horizontal setup")

        # redundant tool warning
        tool = st.get("Tool", "")
        if prev_tool and tool != prev_tool and any(s.get("Tool","")==prev_tool for s in steps if s["Step number"]>sn):
            errors.append(f"Step {sn}: Tool {prev_tool} re‑used later – consider regrouping")
        prev_tool = tool

    # ---- global precedence ordering -------------------------------------
    try:
        from __main__ import validate_precedence  # assumes function exists in notebook scope
        ok_prec, prec_err = validate_precedence(steps)
        if not ok_prec:
            errors.append(prec_err)
    except ImportError:
        pass  # if validate_precedence not available, skip

    if errors:
        return False, "\n".join(errors)
    return True, ""



# ------------------------ FINAL VALIDATION ------------------------------------------
def validate_loop(response_1: str, response_2: str) -> bool:
    try:
        data1 = json.loads(response_1)
    except json.JSONDecodeError as e:
        print(f"Call 1 JSON error: {e}")
        return False

    try:
        data2 = json.loads(response_2)
    except json.JSONDecodeError as e:
        print(f"Call 2 JSON error: {e}")
        return False

    # --- Check response_1 minimal structure ---
    if "Operations" not in data1 or not isinstance(data1["Operations"], list):
        print("Call 1 is missing 'Operations' or it's not a list.")
        return False

    if not data1["Operations"]:
        print("Call 1: 'Operations' list is empty.")
        return False

    # --- Check response_2 minimal structure ---
    if "Setup plan" not in data2 or not isinstance(data2["Setup plan"], list):
        print("Call 2 is missing 'Setup plan' or it's not a list.")
        return False

    if "Process plan" not in data2 or not isinstance(data2["Process plan"], list):
        print("Call 2 is missing 'Process plan' or it's not a list.")
        return False

    if not data2["Process plan"]:
        print("Call 2: 'Process plan' list is empty.")
        return False

    # --- Passed fallback checks ---
    print("Loop validation: minimal required structure present in both responses.")
    return True



# MAIN

In [ ]:
# ---------------------  INITIALIZATION --------------------------------------
is_loop_correct = False
is_call1_correct = False
is_call2_correct = False
is_call3_correct = False

count_loop = 0
count_call1 = 0
count_call2 = 0
count_call3 = 0

max_loop = 3
max_call1 = 5
max_call2 = 5
max_call3 = 5

error_hist_1 = []
error_hist_2 = []
error_hist_3 = []

temperature_1 = 0.1
temperature_2 = 0.3
temperature_3 = 0


def validate_call3(response_str: str):
    """
    Controlla che il JSON di processo (Call 3) sia formalmente valido
    e che il campo 'Position' rispetti le regole:
        • dict con X-Y-Z
        • 'F<ID>' se il pezzo è stato descritto in JSON
        • qualunque stringa non vuota se part_info è testo libero
    Ritorna: (bool_ok, error_string)
    """
    errors = []

    # ---------- JSON valido? ----------
    try:
        plan = json.loads(response_str)
    except json.JSONDecodeError as e:
        return False, f"Plan is not valid JSON: {e}"

    # ---------- check di ogni step ----------
    for step in plan.get("Process plan", []):
        num = step.get("Step number", "N/A")
        pos = step.get("Position")

        # 1) dict con coordinate
        if isinstance(pos, dict):
            if not all(k in pos for k in ("X", "Y", "Z")):
                errors.append(f"Step {num}: Position dict must contain X, Y, Z")

        # 2) stringa
        elif isinstance(pos, str):
            if TEXT_MODE:                           # descrizione in linguaggio naturale
                if not pos.strip():
                    errors.append(f"Step {num}: Position string is empty")
            else:                                   # descrizione in JSON
                if not pos.strip().startswith("F"):
                    errors.append(f"Step {num}: Position '{pos}' must start with 'F'")

        # 3) altro tipo → errore
        else:
            errors.append(f"Step {num}: Position format not recognised")

    return (len(errors) == 0, "\n".join(errors))


# -------------------------------- WORKING LOOP -------------------------------------

while not is_loop_correct and count_loop < max_loop:
    count_loop += 1
    error_hist_1.clear()
    error_hist_2.clear()
    error_hist_3.clear()

    print(f"Loop {count_loop}/{max_loop}")

    # ------------------ CALL 1: OPERATION SELECTION ------------------
    while not is_call1_correct and count_call1 < max_call1:
        count_call1 += 1
        print(f"Call 1 (Operations) {count_call1}/{max_call1}")
        message1 = (
                      system_message1
                    + build_prompt1(part_info)
                    + "\n\n Previous errors:\n" + "\n\n".join(error_hist_1)
                    )
        token_check(message1)
        response_1 = call_api(message1,temperature_1)
        response_1 = clean_llm_json_response(response_1)
        is_call1_correct, error_1 = validate_call1(response_1)

        if not is_call1_correct:
            error_hist_1.append(error_1)
            print("Call 1 failed.")
            print(error_1)
        else:
            print("Call 1 OK.")

    # ------------------ CALL 2: SETUP + PROCESS PLAN ------------------
    while not is_call2_correct and count_call2 < max_call2 and is_call1_correct:
        count_call2 += 1
        print(f"Call 2 (Initial Plan) {count_call2}/{max_call2}")
        try:
            parsed_response_1 = json.loads(response_1)
        except json.JSONDecodeError as e:
            print("JSON decoding error in response_1:", e)
            break
        message2 = (
                    system_message2
                    + build_prompt2(part_info, parsed_response_1)
                    + "\n\n Previous errors:\n" + "\n\n".join(error_hist_2)
                    )

        token_check(message2)
        response_2 = call_api(message2, temperature_2)
        response_2 = clean_llm_json_response(response_2)
        is_call2_correct, error_2 = validate_call2(response_2)

        if not is_call2_correct:
            error_hist_2.append(error_2)
            print("Call 2 failed.")
            print(error_2)
        else:
            print("Call 2 OK.")

    # ------------------ CALL 3: SETUP VALIDATION AND FIX ------------------
    while not is_call3_correct and count_call3 < max_call3 and is_call2_correct:
        count_call3 += 1
        print(f"Call 3 (Geometry Setup Check) {count_call3}/{max_call3}")
        try:
            parsed_response_2 = json.loads(response_2)
        except json.JSONDecodeError as e:
            print("JSON decoding error in response_2:", e)
            break

        # usa solo l'ultimo errore, non tutta la lista
        last_err3 = ("\n\nPrevious error:\n" + error_hist_3[-1]) if error_hist_3 else ""

        message3 = (
                system_message3
                + build_prompt3(part_info, parsed_response_2)
                + last_err3
        )

        token_check(message3)
        response_3 = call_api(message3,temperature_3)
        response_3 = clean_llm_json_response(response_3)
        is_call3_correct, error_3 = validate_call3(response_3)  # Reuse existing validator

        if not is_call3_correct:
            error_hist_3.append(error_3)
            print("Call 3 failed.")
            #print(response_3)
            print(error_3)
        else:
            print("Call 3 OK.")

    # ------------------ FINAL CHECK ------------------
    if is_call1_correct and is_call2_correct and is_call3_correct:
        is_loop_correct = validate_loop(response_1, response_3)
        if not is_loop_correct:
            print("Final combined validation failed.")
    else:
        message4 = (
            system_message4
            + "\n\n=== Last AI Response (response_3) ===\n"
            + response_3
            + "\n\n=== Validation Errors ===\n"
            + "\n".join(error_hist_3)
            + "\n\nIf you can, please infer what design, geometry, or planning issue might be causing the failure, "
              "and suggest how to fix it. Do not generate a new plan."
        )
        error_message = call_api(message4, temperature_2)
        print(error_message)


Loop 1/3
Call 1 (Operations) 1/5
Number of tokens: 1468
Call 1 OK.
Call 2 (Initial Plan) 1/5
Number of tokens: 1383
Call 2 failed.
Setup 1, Step 1: 'Slot Milling' appears before any 'Pocket Milling'.
Call 2 (Initial Plan) 2/5
Number of tokens: 1402
Call 2 OK.
Call 3 (Geometry Setup Check) 1/5
Number of tokens: 1137
Call 3 OK.
Loop validation: minimal required structure present in both responses.


In [ ]:
print(response_1)
print("\n")
print("\n")
print("\n")
print(response_2)
print("\n")
print("\n")
print("\n")
print(response_3)

#print(error_message)

{
  "Operations": [
    {
      "Operation": "Slot Milling",
      "Tool": "End Mill, diameter: 15",
      "Position": "Two cable-pass channels at X ±40 mm, running parallel to the Y-axis",
      "RPM": "7000",
      "Depth": "6",
      "Coolant": "None",
      "Peck_Drilling": "None",
      "Notes": "Pass 1/1, dir: -Z. Slot depth is within limits."
    },
    {
      "Operation": "Pocket Milling",
      "Tool": "End Mill, diameter: 20",
      "Position": "Identification recess at X -60 mm, Y +80 mm",
      "RPM": "7500",
      "Depth": "0.5",
      "Coolant": "None",
      "Peck_Drilling": "None",
      "Notes": "Pass 1/1, dir: -Z. Shallow depth does not require coolant."
    },
    {
      "Operation": "Drilling (Through Hole)",
      "Tool": "Twist Drill, diameter: 13",
      "Position": "Eight counterbored mounting holes at X -80, -40, +40, +80 mm, Y ±75 mm",
      "RPM": "6000",
      "Depth": "5",
      "Coolant": "None",
      "Peck_Drilling": "None",
      "Notes": "Pass 1/1, d

# Post-Processing

In [ ]:
#-------------------------- OUTPUT PARSING ----------------------------------------
def parse_full_plan_json(text: str) -> List[Dict]:
    """
    Given a JSON string, returns an ordered list of dictionaries:
    - Setup #1
    - All steps associated with Setup #1
    - Setup #2
    - All steps associated with Setup #2
    - ...
    """
    data = json.loads(text)

    setup_plan = data.get("Setup plan", [])
    process_plan = data.get("Process plan", [])

    result: List[Dict] = []

    for setup in sorted(setup_plan, key=lambda x: x["Setup number"]):
        # Firstly add the setup
        setup_entry = {
            "Type": "Setup",
            **setup
        }
        result.append(setup_entry)

        # Add all the steps associated to this setup
        for step in sorted(process_plan, key=lambda x: x["Step number"]):
            if step["Setup number"] == setup["Setup number"]:
                step_entry = {
                    "Type": "Step",
                    **step
                }
                result.append(step_entry)

    return result

def plan_to_dataframe(plan_list: List[Dict]) -> pd.DataFrame:

    df = pd.DataFrame(plan_list)
    df.fillna("-", inplace=True)  # Replace NaN with "-"
    return df


# ------------------------ TABEL CREATION --------------------------------------

plan_list = parse_full_plan_json(response_3)
df_steps = plan_to_dataframe(plan_list)
df_steps  # Show the table, for notebook Jupyter

df_steps.to_excel("nome_file.xlsx", index=False)

<ipython-input-97-c8d615d4b961>:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("-", inplace=True)  # Replace NaN with "-"


# G-code

In [ ]:
# ---------------------------- G-CODE GENERATION ------------------------------------------

def extract_coordinates(position_str):
    """Extracts X, Y, Z from string or list."""
    if isinstance(position_str, list):
        coords = position_str
    elif isinstance(position_str, str):
        coords = list(map(float, re.findall(r"[-+]?\d*\.\d+|\d+", position_str)))
    else:
        coords = [0.0, 0.0]

    x = coords[0] if len(coords) > 0 else 0.0
    y = coords[1] if len(coords) > 1 else 0.0
    z = coords[2] if len(coords) > 2 else None
    return x, y, z

def generate_gcode(data):
    gcode_lines = []
    current_tool = None
    tool_ids = {}
    tool_index = 1

    for row in data:
        if row['Type'] == 'Setup':
            gcode_lines.append(f"(--- Setup {row['Setup number']} ---)")
            if row['Orientation'] != '-':
                gcode_lines.append(f"(Orientation: {row['Orientation']})")
            if row['Fixturing'] != '-':
                gcode_lines.append(f"(Fixturing: {row['Fixturing']})")
            if row['Notes'] != '-':
                gcode_lines.append(f"(Notes: {row['Notes']})")
            gcode_lines += ["G21 (Metric units)", "G90 (Absolute positioning)", "G17 (XY plane)"]

        elif row['Type'] == 'Step':
            operation = row['Operation']
            tool = row['Tool']
            x, y, pos_z = extract_coordinates(row.get('Position', '[0.0, 0.0]'))
            z = float(row.get('Depth', pos_z if pos_z is not None else 0.0))
            feed = float(row.get('Feed', 0.1))

            gcode_lines.append(f"(--- Step {row['Step number']}: {operation} ---)")
            gcode_lines.append(f"(Tool: {tool})")

            if tool != current_tool:
                if tool not in tool_ids:
                    tool_ids[tool] = tool_index
                    tool_index += 1
                tid = tool_ids[tool]
                gcode_lines.append(f"T{tid} M6 (Change to {tool})")
                current_tool = tool

            gcode_lines.append(f"S{int(row['RPM'])} M3")
            gcode_lines.append("G0 Z10.0")
            gcode_lines.append(f"G0 X{x:.2f} Y{y:.2f}")

            if "Drilling" in operation:
                gcode_lines.append(f"G1 Z-{z:.2f} F{feed} (Drill)")
            elif "Pocket" in operation:
                gcode_lines.append(f"G1 Z-{z:.2f} F{feed} (Plunge)")
                gcode_lines.append(f"G1 X{x+10.0:.2f} Y{y+10.0:.2f} F{feed} (Pocket move)")
            elif "Chamfering" in operation:
                gcode_lines.append(f"G1 Z-{z:.2f} F{feed}")
                gcode_lines.append(f"G1 X{x+5.0:.2f} Y{y+5.0:.2f} F{feed}")
            elif "Contour" in operation:
                gcode_lines.append(f"G1 Z-{z:.2f} F{feed}")
                gcode_lines.append(f"G1 X{x+10.0:.2f} Y{y:.2f} F{feed}")

            gcode_lines.append("G0 Z10.0")
            gcode_lines.append("M5")

    gcode_lines.append("M30 (Program end)")
    return gcode_lines

G_code = generate_gcode(plan_list)
for line in G_code:
  print(line)

(--- Setup 1 ---)
(Orientation: Top face up)
(Fixturing: Clamped on the machine bed)
(Notes: Machining operations on the top face.)
G21 (Metric units)
G90 (Absolute positioning)
G17 (XY plane)
(--- Step 1: Pocket Milling ---)
(Tool: End Mill, diameter: 20)
T1 M6 (Change to End Mill, diameter: 20)
S7500 M3
G0 Z10.0
G0 X60.00 Y80.00
G1 Z-0.50 F200.0 (Plunge)
G1 X70.00 Y90.00 F200.0 (Pocket move)
G0 Z10.0
M5
(--- Step 2: Slot Milling ---)
(Tool: End Mill, diameter: 15)
T2 M6 (Change to End Mill, diameter: 15)
S7000 M3
G0 Z10.0
G0 X40.00 Y0.00
G0 Z10.0
M5
(--- Setup 2 ---)
(Orientation: Bottom face up)
(Fixturing: Flipped and clamped on the machine bed)
(Notes: Machining operations on the bottom face.)
G21 (Metric units)
G90 (Absolute positioning)
G17 (XY plane)
(--- Step 3: Boring ---)
(Tool: Boring Head, diameter: 20)
T3 M6 (Change to Boring Head, diameter: 20)
S5000 M3
G0 Z10.0
G0 X0.00 Y0.00
G0 Z10.0
M5
(--- Step 4: Drilling (Through Hole) ---)
(Tool: Twist Drill, diameter: 13)
T4 M6 (

In [ ]:
def extract_coordinates(pos):
    """Ritorna  (x,y,z)  da stringa '[..]' oppure lista/tuple."""
    if isinstance(pos, (list, tuple)):
        coords = pos
    elif isinstance(pos, str):
        coords = list(map(float, re.findall(r"[-+]?\d*\.?\d+", pos)))
    else:
        coords = [0.0, 0.0, 0.0]
    x = coords[0] if len(coords) > 0 else 0.0
    y = coords[1] if len(coords) > 1 else 0.0
    z = coords[2] if len(coords) > 2 else 0.0
    return x, y, z


def load_tool_mapping(tooltable_path):
    import json
    with open(tooltable_path, "r") as f:
        table = json.load(f)

    # Costruisce un dizionario: (tipo, diametro) -> (T-code, nome completo)
    mapping = {}
    for tid, tool in table.items():
        name = tool["name"]
        diameter = tool["diameter"]
        base = name.split("×")[0].split("⌀")[0].strip().lower()
        mapping[(base, diameter)] = (int(tid), name)
    return mapping




def parse_tool_name(tool_name):
    """Estrae tipo e diametro da nomi tipo:
    'Twist Drill ⌀4', 'Twist Drill ×4', 'Twist Drill, diameter: 4' ecc."""
    import re
    match = re.search(r"^(.*?)(?:[×⌀, ]*diameter[: ]*)?(\d+(?:\.\d+)?)$", tool_name.strip(), re.IGNORECASE)
    if match:
        base = match.group(1).strip(" ,")
        diameter = float(match.group(2))
        return base.lower(), diameter
    return tool_name.strip().lower(), None


def rough_gcode(plan, tool_map):
    """
    La MINIMA bozza G-code che generavi già:
    serve come 'scheletro' da mostrare al modello LLM,
    in modo che aggiunga i percorsi reali.
    """
    lines, cur_tool = [], None
    for row in plan:
        if row["Type"] == "Setup":
            lines += [
                f"(--- Setup {row['Setup number']} ---)",
                f"(Orientation: {row['Orientation']})",
                f"G21  G90  G17"
            ]
        else:  # è uno Step
            op, tool_str = row["Operation"], row["Tool"]
            tool_type, tool_diam = parse_tool_name(tool_str)
            found = next(((i + 1, name, diam) for i, (name, diam) in enumerate(tool_map)
              if name == tool_type and diam == tool_diam), None)


            print("DEBUG: looking for tool", (tool_type, tool_diam))
            print("DEBUG: found =", found)

            if found is None:
                raise ValueError(f"Tool '{tool_str}' (parsed as {tool_type}, ⌀{tool_diam}) non trovato nella tool table")
            tid, *full_tool_name = found
            if found is None:
                raise ValueError(f"Tool '{tool_str}' non trovato nella tool table")
            if tool_str != cur_tool:
                lines.append(f"T{tid} M6  (change to {full_tool_name})")
                cur_tool = tool_str
            x, y, _ = extract_coordinates(row["Position"])
            lines += [
                f"(Step {row['Step number']}: {op})",
                f"S{row['RPM']} M3",
                f"G0 Z10.",
                f"G0 X{x:.2f} Y{y:.2f}",
                "...",  # placeholder di lavoro
                "G4 P1  (Pause before next operation)"
            ]
    lines.append("M30")
    return lines


SYSTEM_MSG = """
You are a senior CNC programmer.

◆ INPUT
  • You receive:
      1. A machining plan (list of setups & steps, with tool, RPM, feed, depth).
      2. A **rough** skeleton G-code that already contains tool changes
         and safe moves but NOT the actual cutting path.

◆ TASK
  • For every step, expand the skeleton into fully-detailed G-code
    that could realistically run on a 3-axis VMC (Fanuc-style dialect).
  • Include:
        – safe start (G90 G21 G17, etc.)
        – precise plunges / retracts
        – chip-breaking (peck) cycles for deep drills
        – threading cycles (G84 / G76) when ‘Threading’
        – simple raster or trochoidal paths for pockets/slots
        – coolant on/off (M8 / M9) when utile
        – end of program with M30
  • Keep every setup separated by comments.
  • Use the same T-codes that appear in the skeleton; DO NOT invent new tools.
  • Output **only raw G-code**, no markdown.
"""

def build_user_prompt(plan, skel_lines):
    return (
        "=== MACHINING PLAN (JSON) ===\n"
        f"{json.dumps(plan, indent=2)}\n\n"
        "=== INITIAL SKELETON G-CODE ===\n"
        + "\n".join(skel_lines) +
        "\n\n### PLEASE EXPAND the skeleton as explained."
    )


# ──────────────────────────────────────────────────────────────
# 4) FUNZIONE PRINCIPALE: genera G-code dettagliato via LLM
# ────────────────────────────────────────────────────────────

def generate_detailed_gcode(plan, tool_map):
    skel = rough_gcode(plan, tool_map)  # 1) tua bozza
    prompt = build_user_prompt(plan, skel)  # 2) prompt utente

    response = openai.chat.completions.create(  # 3) chiamata LLM
        model=ENGINE,
        messages=[
            {"role": "system", "content": SYSTEM_MSG.strip()},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    gcode_full = response.choices[0].message.content.strip().splitlines()
    return gcode_full

tooltable_path = "/content/tooltable_camotics_3.json"
with open(tooltable_path, "r") as f:
        table = json.load(f)

print(table)

import re

def extract_tool_type_and_diameter(name):
    """
    Da 'Twist Drill Ø4' o 'End Mill ×10' → ('twist drill', 4)
    """
    match = re.search(r"^(.*?)[×ø⌀]?\s*(\d+(?:\.\d+)?)\s*$", name.lower())
    if match:
        tool_type = match.group(1).strip()
        diameter = int(match.group(2))
        return tool_type, diameter
    return name.lower().strip(), None  # fallback

# Esempio con il tuo dizionario chiamato tool_dict
tool_tuples = [
    extract_tool_type_and_diameter(v["name"])
    for v in table.values()
]

print(tool_tuples)

detailed_gc = generate_detailed_gcode(plan_list,tool_tuples)

# Salva su disco in formato .nc
out_path = "job_output.nc"
with open(out_path, "w") as f:
    for line in detailed_gc:
        f.write(line + "\n")

print(f"✅ Detailed G-code salvato in → {out_path}")
print(detailed_gc)

FileNotFoundError: [Errno 2] No such file or directory: '/content/tooltable_camotics_3.json'

# Automatic Documentation

In [ ]:
!pip install fpdf pandas

from fpdf import FPDF
from datetime import datetime
import unicodedata
from zoneinfo import ZoneInfo


def clean_text(text):
    return unicodedata.normalize("NFKD", str(text)).encode("latin-1", "ignore").decode("latin-1")


def generate_cnc_documentation(part_info, df_steps, gcode_str, output_path="cnc_documentation_AI.pdf"):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Title
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(0, 15, txt="CNC Automatic Documentation", ln=True, align='C')
    pdf.set_font("Arial", '', 10)
    local_time = datetime.now(ZoneInfo("Europe/Rome"))
    pdf.cell(0, 7, txt=f"Generated on {local_time.strftime('%Y-%m-%d %H:%M')}", ln=True)
    pdf.ln(5)

    # 1. Job Overview
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="1. Job Overview", ln=True)
    pdf.set_font("Arial", '', 10)

    if isinstance(part_info, dict):
        overview_text = f"""
        Geometry: {part_info.get('geometry', '-')} {part_info.get('geometry_parameters', '')}
        Material: {part_info.get('material', '-')}
        Reference Origin: {part_info.get('reference_origin', part_info.get('default_origin', '-'))}
        Tolerance: {part_info.get('tolerance', '-')}
        Surface Finish: {part_info.get('surface_finish', '-')}
        """
    elif isinstance(part_info, str):
        prompt = f"""
        You are a CNC process planner assistant. Given the following free-form description of a machined part, generate a short technical summary including:
        - Geometry and key dimensions
        - Material
        - Datum or reference origin
        - Notable machining features
        - Surface finish and tolerances (if any)

        Input:
        {part_info}

        Output:
        """
        overview_text = call_api(prompt, temperature_3)
    else:
        overview_text = "Invalid input format for part_info."

    pdf.multi_cell(0, 7, clean_text(overview_text))

    # 2. Setup Plan
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="2. Setup Plan", ln=True)
    pdf.set_font("Arial", '', 10)
    setups = df_steps[df_steps['Type'] == 'Setup']
    for _, row in setups.iterrows():
        prompt = f"""
        Describe the CNC setup with:
        - Orientation: {row['Orientation']}
        - Fixturing: {row['Fixturing']}
        - Notes: {row['Notes']}
        """
        ai_descr = call_api(prompt, temperature_2)
        pdf.multi_cell(0, 7, clean_text(f"Setup #{row['Setup number']}:\n{ai_descr}"))
        pdf.ln(2)

    # 3. Process Plan
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="3. Process Plan", ln=True)
    pdf.set_font("Arial", '', 10)
    steps = df_steps[df_steps['Type'] == 'Step']
    for _, row in steps.iterrows():
        prompt = f"""
        Describe this machining step:
        - Operation: {row['Operation']}
        - Tool: {row['Tool']}
        - RPM: {row['RPM']}, Feed: {row['Feed']}, Depth: {row['Depth']}
        - Position: {row['Position']}
        - Notes: {row['Notes']}
        """
        ai_descr = call_api(prompt, temperature_2)
        pdf.multi_cell(0, 7, clean_text(f"Step {row['Step number']} – {row['Operation']}:\n{ai_descr}"))
        pdf.ln(2)

    # 4. Tool List
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="4. Tool List", ln=True)
    pdf.set_font("Arial", '', 10)
    tool_set = sorted(set(steps['Tool']))
    for tool in tool_set:
        pdf.cell(0, 7, txt=f"- {clean_text(tool)}", ln=True)

    # 5. G-code
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="5. NC File (G-code)", ln=True)
    pdf.set_font("Courier", size=10)
    for line in gcode_str.strip().splitlines():
        pdf.multi_cell(0, 5, clean_text(line))

    # 6. Safety & Competency
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="6. Safety & Competency Standards", ln=True)
    pdf.set_font("Arial", '', 10)
    prompt = "List important safety rules and competency requirements for operating a CNC machine based on general best practices."
    safety = call_api(prompt, temperature_2)
    pdf.multi_cell(0, 7, clean_text(safety))

    # 7. Maintenance
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, txt="7. Maintenance Procedures", ln=True)
    pdf.set_font("Arial", '', 10)
    prompt = "Give a typical CNC machine maintenance schedule, including daily, weekly, and monthly tasks."
    maintenance = call_api(prompt, temperature_2)
    pdf.multi_cell(0, 7, clean_text(maintenance))

    # Saving
    pdf.output(output_path)
    print(f"AI-enhanced documentation saved to: {output_path}")


gcode_str = "\n".join(G_code)

generate_cnc_documentation(part_info, df_steps, gcode_str)
from google.colab import files
files.download("cnc_documentation_AI.pdf")

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=09ea39c27cd15e8d1c3190232e44406517256ab5e805efac2354b77c63015210
  Stored in directory: /root/.cache/pip/wheels/65/4f/66/bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf
AI-enhanced documentation saved to: cnc_documentation_AI.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>